In [6]:
# ============================================================
# SOIL DATASET PREPROCESSING + AUGMENTATION
#
# TRAIN:
#   All original training photos
#   -> original preprocessing
#   -> exactly 80 augmentations per original photo
#   -> save directly into soil_augmented_dataset/train
#
# NO TRAIN/VALIDATION SPLIT.
#
# TEST:
#   Original preprocessing only
#   -> no augmentation
#   -> save into soil_augmented_dataset/test
#
# Original preprocessing, augmentation families, values, and
# input/output paths are retained.
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import io
import re
import time
import unicodedata
import warnings

from pathlib import Path

from tqdm.auto import tqdm

import cv2
import numpy as np
import pandas as pd

from PIL import Image, ImageOps, ImageEnhance


# ============================================================
# PATHS — UNCHANGED
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    INPUT_DIR = sorted(
        Path("/kaggle/input").glob("**/Training_labels_updated.csv")
    )[0].parent
    OUTPUT_DIR = Path("/kaggle/working")
else:
    INPUT_DIR = Path("input")
    OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# ORIGINAL CONSTANTS — KEPT AS PROVIDED
# ============================================================

SIZES = [
    0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200
]

SIZE_COLS = [
    "0.002", "0.0063", "0.02", "0.063", "0.2", "0.63",
    "2", "6.3", "20", "63", "200"
]

W_EMD = np.diff(np.log10(SIZES))

CAM_EDGE, CAM_A52 = ("Motorola Edge", "Samsung A52")

CFG = {
    "target_ppm": 4.6,
    "crop_frac": 0.5,
    "dog_sigmas_mm": [0.25, 0.5, 1.0, 2.0, 4.0, 8.0],
    "train_like_above_ppm": 6.0,
    "ridge_alpha": 1.0,
    "weights_group": {"gravel": 0.2, "non_gravel": 0.8},

    # Weibull label fit
    "grid_lnb": [-6.0, 4.0, 41],
    "grid_c": [0.1, 2.0, 39],
    "c_floor": 0.02,

    # SAM stone measurement settings — retained
    "sam_model": "facebook/sam-vit-base",
    "sam_min_ppm": 12.0,
    "work_ppm": 2.0,
    "work_crop": 0.12,
    "pred_iou_thresh": 0.86,
    "stability_score_thresh": 0.9,
    "points_per_batch": 64,
    "overlap_drop": 0.5,
    "min_fill": 0.35,
    "min_side_mm": 8.0,
    "max_area_frac": 0.5,
    "apply_if_A63_ge": 0.02,

    # Augmentation settings — retained
    "augmented_per_image": 80,
    "brightness_decrease": 0.85,
    "brightness_increase": 1.15,
    "brightness_decrease_strong": 0.70,
    "brightness_increase_strong": 1.30,
    "saturation_decrease": 0.85,
    "saturation_increase": 1.15,
    "saturation_decrease_strong": 0.70,
    "saturation_increase_strong": 1.30,
    "output_jpeg_quality": 95,
}

print("input:", INPUT_DIR)

# ============================================================
# CAMERA PARSING — ORIGINAL
# ============================================================

CAMERA_RE = re.compile(
    r"^(Motorola_Edge_60_fusion|Motorola_Edge|"
    r"Samsung_A52|iPhone_?14|iPhone_?16)_"
    r"(.+?)(?:_\d+| \(\d+\))?$"
)

CAMERA_KEY = {
    "Motorola_Edge_60_fusion": "Motorola Edge 60 Fusion",
    "Motorola_Edge": CAM_EDGE,
    "Samsung_A52": CAM_A52,
    "iPhone14": "iPhone 14",
    "iPhone_14": "iPhone 14",
    "iPhone16": "iPhone 16",
    "iPhone_16": "iPhone 16",
}


def normalize_id(raw: str) -> str:
    """Map the sample part of a filename to submission ID spelling."""
    s = unicodedata.normalize("NFC", raw)
    s = s.replace("ü", "ue").replace("ö", "oe").replace("ä", "ae")
    return s.replace(",", "_")


# ============================================================
# LOAD ORIGINAL DATA — UNCHANGED
# ============================================================

labels = pd.read_csv(
    INPUT_DIR / "Training_labels_updated.csv",
    dtype={"sample_id": str},
).set_index("sample_id")

sample_sub = pd.read_csv(
    INPUT_DIR / "sample_submission.csv",
    dtype={"sample_id": str},
)

ppm_tab = pd.read_csv(INPUT_DIR / "ppm_updated.csv").set_index("phone")

# ============================================================
# BUILD PHOTO TABLE — UNCHANGED
# ============================================================

rows = []

for split, pattern in [
    ("train", "Training-All_Photos*"),
    ("test", "Test_All_Photos*"),
]:
    for p in sorted(INPUT_DIR.glob(f"{pattern}/**/*")):
        if p.suffix.lower() not in {".jpg", ".jpeg"}:
            continue

        m = CAMERA_RE.match(unicodedata.normalize("NFC", p.stem))
        if m is None:
            raise ValueError(f"unexpected file name: {p.name}")

        with Image.open(p) as im:
            w, h = im.size

        rows.append({
            "split": split,
            "path": p,
            "camera": CAMERA_KEY[m.group(1)],
            "sample_id": normalize_id(m.group(2)),
            "w": w,
            "h": h,
        })

photos = pd.DataFrame(rows)

photos["ppm_eff"] = [
    ppm_tab.loc[c, "ppm"] * max(w, h) / ppm_tab.loc[c, "width"]
    for c, w, h in zip(
        photos.camera, photos.w, photos.h, strict=True
    )
]

# ============================================================
# ORIGINAL ASSERTIONS — UNCHANGED
# ============================================================

assert sorted(set(photos.query("split == 'train'").sample_id)) == sorted(labels.index)
assert set(photos.query("split == 'test'").sample_id) == set(sample_sub.sample_id)

# ============================================================
# SOIL TYPE — ORIGINAL
# ============================================================

lab = labels.copy()

lab["type"] = np.select(
    [
        lab["0.063"] >= 40,
        (lab["63"] - lab["2"]) >= 40,
    ],
    ["fine", "gravel"],
    "sand",
)

lab["group"] = np.where(lab["type"] == "gravel", "gravel", "non_gravel")

print(
    photos.groupby(["split", "camera"]).agg(
        photos=("path", "size"),
        samples=("sample_id", "nunique"),
        px_per_mm=("ppm_eff", "median"),
    ).round(1)
)

# ============================================================
# TRAIN-LIKE SCALE — ORIGINAL
# ============================================================

_train_small = photos[
    (photos.split == "train")
    & (photos.ppm_eff <= CFG["train_like_above_ppm"])
]

if len(_train_small) == 0:
    raise RuntimeError(
        "No training photos meet the train-like scale condition."
    )

TRAIN_LIKE_PPM = float(_train_small.ppm_eff.median())

_train_a52 = sorted(_train_small[_train_small.camera == CAM_A52].path)
if not _train_a52:
    raise RuntimeError(
        "No Samsung A52 training photo is available to copy JPEG quantisation tables."
    )

with Image.open(_train_a52[0]) as _qimage:
    TRAIN_QTABLES = _qimage.quantization

print(f"train-like target: {TRAIN_LIKE_PPM:.3f} px/mm")

# ============================================================
# TRAIN-LIKE IMAGE DEGRADATION — ORIGINAL
# ============================================================

def to_train_like(
    rgb: np.ndarray,
    ppm_eff: float,
) -> tuple[np.ndarray, float]:
    """
    Downscale a sharp photo to the training scale and re-save it
    as JPEG with the training q-tables (4:2:0).
    """
    f = TRAIN_LIKE_PPM / ppm_eff

    small = cv2.resize(
        rgb,
        (round(rgb.shape[1] * f), round(rgb.shape[0] * f)),
        interpolation=cv2.INTER_AREA,
    )

    buf = io.BytesIO()
    Image.fromarray(small).save(
        buf,
        format="JPEG",
        qtables=TRAIN_QTABLES,
        subsampling=2,
    )

    return (
        np.asarray(
            Image.open(io.BytesIO(buf.getvalue())).convert("RGB")
        ),
        TRAIN_LIKE_PPM,
    )

# ============================================================
# LOAD FUNCTION — ORIGINAL PREPROCESSING
# ============================================================

def load_rgb(path: Path, ppm_eff: float) -> np.ndarray:
    """
    EXIF rotation
    -> train-like path if sharp
    -> RGB
    -> landscape
    -> 4.6 px/mm
    -> centre crop
    """
    with Image.open(path) as source:
        im = ImageOps.exif_transpose(source).copy()

    if ppm_eff > CFG["train_like_above_ppm"]:
        rgb, ppm_eff = to_train_like(np.asarray(im.convert("RGB")), ppm_eff)
        im = Image.fromarray(rgb)
    else:
        im = im.convert("RGB")

    a = np.asarray(im)

    if a.shape[0] > a.shape[1]:
        a = np.ascontiguousarray(np.rot90(a))

    f = CFG["target_ppm"] / ppm_eff

    a = cv2.resize(
        a,
        (round(a.shape[1] * f), round(a.shape[0] * f)),
        interpolation=cv2.INTER_AREA,
    )

    h, w = a.shape[:2]
    ch = int(h * CFG["crop_frac"])
    cw = int(w * CFG["crop_frac"])
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2

    return a[y0:y0 + ch, x0:x0 + cw].astype(np.uint8)

# ============================================================
# GEOMETRIC AUGMENTATIONS — ORIGINAL
# ============================================================

def geometric_augmentations(
    image: Image.Image,
) -> list[tuple[str, Image.Image]]:
    variants = []

    # No flip + rotations
    variants.append(("rot0", image.copy()))
    variants.append(("rot90", image.rotate(90, expand=False)))
    variants.append(("rot180", image.rotate(180, expand=False)))
    variants.append(("rot270", image.rotate(270, expand=False)))

    # Horizontal flip + rotations
    hflip = ImageOps.mirror(image)
    variants.append(("hflip_rot0", hflip.copy()))
    variants.append(("hflip_rot90", hflip.rotate(90, expand=False)))
    variants.append(("hflip_rot180", hflip.rotate(180, expand=False)))
    variants.append(("hflip_rot270", hflip.rotate(270, expand=False)))

    assert len(variants) == 8
    return variants

# ============================================================
# PHOTOMETRIC AUGMENTATIONS — ORIGINAL
# ============================================================

def photometric_augmentations(
    image: Image.Image,
) -> list[tuple[str, Image.Image]]:
    variants = []

    variants.append(("original", image.copy()))

    variants.append((
        "brightness_up",
        ImageEnhance.Brightness(image).enhance(CFG["brightness_increase"]),
    ))

    variants.append((
        "brightness_down",
        ImageEnhance.Brightness(image).enhance(CFG["brightness_decrease"]),
    ))

    variants.append((
        "brightness_up_strong",
        ImageEnhance.Brightness(image).enhance(
            CFG["brightness_increase_strong"]
        ),
    ))

    variants.append((
        "brightness_down_strong",
        ImageEnhance.Brightness(image).enhance(
            CFG["brightness_decrease_strong"]
        ),
    ))

    variants.append((
        "saturation_up",
        ImageEnhance.Color(image).enhance(CFG["saturation_increase"]),
    ))

    variants.append((
        "saturation_down",
        ImageEnhance.Color(image).enhance(CFG["saturation_decrease"]),
    ))

    variants.append((
        "saturation_up_strong",
        ImageEnhance.Color(image).enhance(
            CFG["saturation_increase_strong"]
        ),
    ))

    variants.append((
        "saturation_down_strong",
        ImageEnhance.Color(image).enhance(
            CFG["saturation_decrease_strong"]
        ),
    ))

    combined = ImageEnhance.Brightness(image).enhance(
        CFG["brightness_increase"]
    )
    combined = ImageEnhance.Color(combined).enhance(
        CFG["saturation_increase"]
    )
    variants.append(("brightness_up_saturation_up", combined))

    assert len(variants) == 10
    return variants

# ============================================================
# CREATE EXACTLY 80 VARIANTS — ORIGINAL
# ============================================================

def create_80_augmented_images(
    rgb: np.ndarray,
) -> list[tuple[str, Image.Image]]:
    base = Image.fromarray(rgb).convert("RGB")
    geometric = geometric_augmentations(base)
    all_variants = []

    for geometric_name, geometric_image in geometric:
        photometric = photometric_augmentations(geometric_image)

        for photometric_name, photometric_image in photometric:
            name = f"{geometric_name}__{photometric_name}"
            all_variants.append((name, photometric_image))

    assert len(all_variants) == 80
    return all_variants

# ============================================================
# SAFE FILE-NAME FUNCTION — ORIGINAL
# ============================================================

def safe_name(value: str) -> str:
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value.strip("_")

# ============================================================
# SELECT ALL ORIGINAL TRAIN PHOTOS
#
# NO TRAIN/VALIDATION SPLIT.
# All 80 variants of every training photo go to TRAIN_DIR.
# ============================================================

original_train = photos[photos.split == "train"].copy()
test_rows = photos[photos.split == "test"].copy()

DATASET_DIR = OUTPUT_DIR / "soil_augmented_dataset"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

print()
print("=" * 80)
print("TRAIN AUGMENTATION — NO TRAIN/VALIDATION SPLIT")
print("=" * 80)
print("Original training photos:", len(original_train))
print("Unique training samples:", original_train.sample_id.nunique())
print("Augmentations per original photo:", CFG["augmented_per_image"])
print(
    "Expected augmented training images:",
    len(original_train) * CFG["augmented_per_image"],
)
print("Training output:", TRAIN_DIR)
print("Test output:", TEST_DIR)

# ============================================================
# STEP 1 — AUGMENT ALL TRAINING PHOTOS
# ============================================================

t0 = time.time()
all_augmented_count = 0
per_original_counts = {}

for row in tqdm(
    original_train.itertuples(),
    total=len(original_train),
    desc="Augmenting training photos",
    unit="photo",
):
    rgb = load_rgb(row.path, row.ppm_eff)
    variants = create_80_augmented_images(rgb)

    sample_dir = TRAIN_DIR / safe_name(row.sample_id)
    image_dir = sample_dir / safe_name(row.path.stem)
    image_dir.mkdir(parents=True, exist_ok=True)

    # Remove stale JPEG outputs for this original photo so reruns
    # do not leave unexpected files in its output folder.
    for old_file in image_dir.glob("*.jpg"):
        old_file.unlink()

    for variant_number, (variant_name, variant_image) in enumerate(
        variants, start=1
    ):
        output_path = image_dir / (
            f"{variant_number:03d}__{variant_name}.jpg"
        )

        variant_image.save(
            output_path,
            format="JPEG",
            quality=CFG["output_jpeg_quality"],
            subsampling=0,
        )

        all_augmented_count += 1

    per_original_counts[(row.sample_id, row.path.stem)] = len(
        list(image_dir.glob("*.jpg"))
    )

print(f"\nTraining augmentation time: {time.time() - t0:.2f} s")

# ============================================================
# VERIFY TRAIN AUGMENTATIONS
# ============================================================

expected_train_count = len(original_train) * CFG["augmented_per_image"]

print()
print("=" * 80)
print("TRAIN AUGMENTATION CHECK")
print("=" * 80)
print("Expected augmented images:", expected_train_count)
print("Generated during this run:", all_augmented_count)

assert all_augmented_count == expected_train_count

for (sample_id, photo_stem), count in per_original_counts.items():
    assert count == CFG["augmented_per_image"], (
        f"Expected {CFG['augmented_per_image']} variants for "
        f"{sample_id}/{photo_stem}; found {count}"
    )

actual_train_count = sum(
    1 for p in TRAIN_DIR.rglob("*.jpg") if p.is_file()
)

print("JPEG files currently in train directory:", actual_train_count)

# Existing output folders are not globally cleared. If a previous run
# used a different source set, old sample/photo folders can remain.
# The count below is reported separately from this run's exact count.

# ============================================================
# STEP 2 — PREPROCESS TEST PHOTOS WITHOUT AUGMENTATION
# ============================================================

print()
print("=" * 80)
print("PREPROCESSING TEST PHOTOS — NO AUGMENTATION")
print("=" * 80)

t0 = time.time()
test_count = 0

for row in tqdm(
    test_rows.itertuples(),
    total=len(test_rows),
    desc="Processing test photos",
    unit="photo",
):
    rgb = load_rgb(row.path, row.ppm_eff)
    image = Image.fromarray(rgb).convert("RGB")

    sample_dir = TEST_DIR / safe_name(row.sample_id)
    sample_dir.mkdir(parents=True, exist_ok=True)

    output_path = sample_dir / f"{safe_name(row.path.stem)}.jpg"

    image.save(
        output_path,
        format="JPEG",
        quality=CFG["output_jpeg_quality"],
        subsampling=0,
    )

    test_count += 1

print(f"Test preprocessing time: {time.time() - t0:.2f} s")

# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("FINAL DATASET SUMMARY")
print("=" * 80)
print("Original training photos:", len(original_train))
print("Unique training samples:", original_train.sample_id.nunique())
print("Augmentations per original training photo:", CFG["augmented_per_image"])
print("Expected augmented training images:", expected_train_count)
print("Generated augmented training images:", all_augmented_count)
print("Test photos processed without augmentation:", test_count)
print("Validation split:", "NONE")
print("DoG features calculated:", "NO")

print()
print("Training directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

assert all_augmented_count == expected_train_count
assert test_count == len(test_rows)

print()
print("=" * 80)
print("DONE")
print("=" * 80)
print("All original training photos were augmented.")
print("No train/validation split was performed.")
print("All training augmentations are saved under train/<sample>/<photo>/.")
print("Test photos were preprocessed and saved without augmentation.")


input: /kaggle/input/competitions/soil-grain-size-from-photos
                               photos  samples  px_per_mm
split camera                                             
test  iPhone 14                    14        3       13.9
      iPhone 16                    21        7       19.5
train Motorola Edge                69       22        4.6
      Motorola Edge 60 Fusion       3        1       12.5
      Samsung A52                  55       22        4.6
train-like target: 4.597 px/mm

TRAIN AUGMENTATION — NO TRAIN/VALIDATION SPLIT
Original training photos: 127
Unique training samples: 24
Augmentations per original photo: 80
Expected augmented training images: 10160
Training output: /kaggle/working/soil_augmented_dataset/train
Test output: /kaggle/working/soil_augmented_dataset/test


Augmenting training photos:   0%|          | 0/127 [00:00<?, ?photo/s]


Training augmentation time: 64.00 s

TRAIN AUGMENTATION CHECK
Expected augmented images: 10160
Generated during this run: 10160
JPEG files currently in train directory: 10160

PREPROCESSING TEST PHOTOS — NO AUGMENTATION


Processing test photos:   0%|          | 0/35 [00:00<?, ?photo/s]

Test preprocessing time: 13.52 s

FINAL DATASET SUMMARY
Original training photos: 127
Unique training samples: 24
Augmentations per original training photo: 80
Expected augmented training images: 10160
Generated augmented training images: 10160
Test photos processed without augmentation: 35
Validation split: NONE
DoG features calculated: NO

Training directory: /kaggle/working/soil_augmented_dataset/train
Test directory: /kaggle/working/soil_augmented_dataset/test

DONE
All original training photos were augmented.
No train/validation split was performed.
All training augmentations are saved under train/<sample>/<photo>/.
Test photos were preprocessed and saved without augmentation.


In [9]:
# ============================================================
# DINOv2 ViT-L/14
# FROZEN GLOBAL + LOCAL FEATURE EXTRACTION
#
# ACTUAL DATASET STRUCTURE:
#
#   TRAIN:
#       127 original photos × 80 augmentations
#       = 10,160 augmented images
#
#   VALIDATION:
#       NONE
#
#   TEST:
#       35 preprocessed, non-augmented photos
#
# OUTPUT REPRESENTATIONS:
#
#   CLS:
#       1024-D global representation
#
#   MEAN_PATCH:
#       1024-D local representation
#
#   GLOBAL_LOCAL:
#       2048-D [CLS + mean_patch]
#
# OUTPUT:
#
#   /kaggle/working/dinov2_l_features/train/
#   /kaggle/working/dinov2_l_features/test/
#
# NO DINO TRAINING
# NO RANDOM AUGMENTATION
# NO TRAIN/VALIDATION SPLIT
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import gc
import time
import random
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


# ============================================================
# CONFIGURATION
# ============================================================

CFG_DINO = {

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    "dataset_dir":
        "/kaggle/working/soil_augmented_dataset",

    # --------------------------------------------------------
    # Feature output
    # --------------------------------------------------------

    "feature_dir":
        "/kaggle/working/dinov2_l_features",

    # --------------------------------------------------------
    # DINO input resolution
    #
    # 392 = 28 * 14
    # Patch grid = 28 x 28 = 784 tokens
    # --------------------------------------------------------

    "image_size":
        392,

    # --------------------------------------------------------
    # Batch
    # --------------------------------------------------------

    "batch_size":
        16,

    "num_workers":
        2,

    "pin_memory":
        True,

    "persistent_workers":
        True,

    # --------------------------------------------------------
    # Mixed precision inference
    # --------------------------------------------------------

    "use_amp":
        True,

    # --------------------------------------------------------
    # Feature storage
    # --------------------------------------------------------

    "save_dtype":
        "float32"
}


# ============================================================
# PATHS
# ============================================================

DATASET_DIR = Path(
    CFG_DINO["dataset_dir"]
)

TRAIN_DIR = (
    DATASET_DIR / "train"
)

TEST_DIR = (
    DATASET_DIR / "test"
)

FEATURE_DIR = Path(
    CFG_DINO["feature_dir"]
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("DINOv2 ViT-L/14 FEATURE EXTRACTION")
print("=" * 80)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ============================================================
# VERIFY DATASET DIRECTORIES
# ============================================================

assert DATASET_DIR.exists(), (
    f"Dataset directory not found: {DATASET_DIR}"
)

assert TRAIN_DIR.exists(), (
    f"Training directory not found: {TRAIN_DIR}"
)

assert TEST_DIR.exists(), (
    f"Test directory not found: {TEST_DIR}"
)

print()
print("Training directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)
print("Feature directory:", FEATURE_DIR)


# ============================================================
# IMAGE EXTENSIONS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png"
}


# ============================================================
# DISCOVER TRAINING AUGMENTATIONS
#
# Structure:
#
# train/
#     sample_id/
#         original_photo_1/
#             001__rot0__original.jpg
#             ...
#             080__...
#
#         original_photo_2/
#             001__rot0__original.jpg
#             ...
#             080__...
#
# Every augmentation inherits the original sample_id.
# ============================================================

def discover_augmented_files(root):

    files = []

    for sample_dir in sorted(root.iterdir()):

        if not sample_dir.is_dir():
            continue

        sample_id = sample_dir.name

        for original_dir in sorted(sample_dir.iterdir()):

            if not original_dir.is_dir():
                continue

            for image_path in sorted(
                original_dir.iterdir()
            ):

                if (
                    image_path.suffix.lower()
                    not in IMAGE_EXTENSIONS
                ):
                    continue

                files.append(
                    (
                        image_path,
                        sample_id
                    )
                )

    return files


# ============================================================
# DISCOVER ORIGINAL TEST PHOTOS
#
# Structure:
#
# test/
#     sample_id/
#         original_photo.jpg
#
# Test images are not augmented.
# ============================================================

def discover_original_files(root):

    files = []

    for sample_dir in sorted(root.iterdir()):

        if not sample_dir.is_dir():
            continue

        sample_id = sample_dir.name

        for image_path in sorted(
            sample_dir.iterdir()
        ):

            if (
                image_path.suffix.lower()
                not in IMAGE_EXTENSIONS
            ):
                continue

            files.append(
                (
                    image_path,
                    sample_id
                )
            )

    return files


# ============================================================
# DISCOVER DATA
# ============================================================

print()
print("=" * 80)
print("DISCOVERING IMAGES")
print("=" * 80)

train_files = discover_augmented_files(
    TRAIN_DIR
)

test_files = discover_original_files(
    TEST_DIR
)


# ============================================================
# EXPECTED COUNTS
# ============================================================

EXPECTED_TRAIN_IMAGES = 10160
EXPECTED_TEST_IMAGES = 35

print()
print("Training augmented images:", len(train_files))
print("Test original images:", len(test_files))

assert len(train_files) == EXPECTED_TRAIN_IMAGES, (
    f"Expected {EXPECTED_TRAIN_IMAGES} training images, "
    f"found {len(train_files)}"
)

assert len(test_files) == EXPECTED_TEST_IMAGES, (
    f"Expected {EXPECTED_TEST_IMAGES} test images, "
    f"found {len(test_files)}"
)


# ============================================================
# VERIFY TRAINING SAMPLE DISTRIBUTION
# ============================================================

train_sample_ids = sorted(
    set(
        sample_id
        for _, sample_id in train_files
    )
)

test_sample_ids = sorted(
    set(
        sample_id
        for _, sample_id in test_files
    )
)

print()
print("Unique training physical samples:", len(train_sample_ids))
print("Unique test physical samples:", len(test_sample_ids))


# There are 24 unique training samples.
assert len(train_sample_ids) == 24, (
    f"Expected 24 training samples, "
    f"found {len(train_sample_ids)}"
)


# ============================================================
# VERIFY 80 AUGMENTATIONS PER ORIGINAL PHOTO
# ============================================================

print()
print("=" * 80)
print("VERIFYING TRAINING AUGMENTATIONS")
print("=" * 80)

train_original_dirs = []

for sample_dir in sorted(TRAIN_DIR.iterdir()):

    if not sample_dir.is_dir():
        continue

    for original_dir in sorted(sample_dir.iterdir()):

        if not original_dir.is_dir():
            continue

        image_count = sum(
            1
            for image_path in original_dir.iterdir()
            if image_path.suffix.lower() in IMAGE_EXTENSIONS
        )

        assert image_count == 80, (
            f"Expected 80 images in {original_dir}, "
            f"found {image_count}"
        )

        train_original_dirs.append(original_dir)


assert len(train_original_dirs) == 127, (
    f"Expected 127 original-photo groups, "
    f"found {len(train_original_dirs)}"
)

assert (
    len(train_original_dirs) * 80
    == len(train_files)
)

print("Original training photos:", len(train_original_dirs))
print("Augmentations per photo: 80")
print("Verified training images:", len(train_files))


# ============================================================
# DINO IMAGE TRANSFORM
#
# Keep the existing preprocessing unchanged:
#
# Resize -> ToTensor -> ImageNet normalization
#
# DINOv2 ViT-L/14 embedding dimension = 1024
# ============================================================

IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)

dino_transform = transforms.Compose(
    [

        transforms.Resize(
            (
                CFG_DINO["image_size"],
                CFG_DINO["image_size"]
            ),
            interpolation=(
                transforms.InterpolationMode.BICUBIC
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )

    ]
)


# ============================================================
# DATASET
# ============================================================

class DINOImageDataset(Dataset):

    def __init__(self, files, transform):

        self.files = files
        self.transform = transform

    def __len__(self):

        return len(self.files)

    def __getitem__(self, index):

        image_path, sample_id = self.files[index]

        with Image.open(image_path) as image:

            image = image.convert("RGB").copy()

        image = self.transform(image)

        return (
            image.contiguous(),
            sample_id,
            str(image_path)
        )


# ============================================================
# COLLATE
# ============================================================

def safe_collate(batch):

    images = torch.stack(
        [item[0] for item in batch],
        dim=0
    )

    sample_ids = [
        item[1] for item in batch
    ]

    paths = [
        item[2] for item in batch
    ]

    return images, sample_ids, paths


# ============================================================
# DATALOADER
# ============================================================

num_workers = min(
    CFG_DINO["num_workers"],
    os.cpu_count() or 1
)

persistent_workers = (
    CFG_DINO["persistent_workers"]
    and num_workers > 0
)


def make_loader(files):

    dataset = DINOImageDataset(
        files,
        dino_transform
    )

    return DataLoader(
        dataset,

        batch_size=CFG_DINO["batch_size"],

        shuffle=False,

        num_workers=num_workers,

        pin_memory=(
            CFG_DINO["pin_memory"]
            and DEVICE.type == "cuda"
        ),

        persistent_workers=persistent_workers,

        collate_fn=safe_collate,

        drop_last=False
    )


train_loader = make_loader(train_files)
test_loader = make_loader(test_files)


# ============================================================
# LOAD FROZEN DINOv2 ViT-L/14
# ============================================================

print()
print("=" * 80)
print("LOADING DINOv2 ViT-L/14")
print("=" * 80)

t0 = time.time()

dino = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vitl14"
)

dino = dino.to(DEVICE)

dino.eval()

for parameter in dino.parameters():
    parameter.requires_grad = False

print("DINOv2 loaded.")
print(f"Load time: {time.time() - t0:.1f} seconds")


# ============================================================
# VERIFY DINO OUTPUT
# ============================================================

print()
print("=" * 80)
print("VERIFYING DINO OUTPUT")
print("=" * 80)

test_batch = next(iter(train_loader))

test_images = test_batch[0].to(
    DEVICE,
    non_blocking=True
)

with torch.no_grad():

    test_output = dino(
        test_images
    )

print(
    "Standard DINO output shape:",
    test_output.shape
)

assert test_output.shape[1] == 1024, (
    "Expected DINOv2 ViT-L/14 embedding dimension 1024."
)

del test_batch, test_images, test_output

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# FEATURE EXTRACTION
#
# GLOBAL:
#     x_norm_clstoken
#     1024 dimensions
#
# LOCAL:
#     Mean of x_norm_patchtokens
#     1024 dimensions
#
# GLOBAL_LOCAL:
#     Concatenation [CLS, mean_patch]
#     2048 dimensions
#
# No gradient computation and no model training.
# ============================================================

@torch.no_grad()
def extract_features(loader, split_name):

    global_features = []
    local_features = []
    combined_features = []

    sample_ids = []
    image_paths = []

    progress = tqdm(
        loader,
        desc=f"DINO {split_name}",
        unit="batch"
    )

    for images, ids, paths in progress:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # DINO forward pass
        # ----------------------------------------------------

        if (
            CFG_DINO["use_amp"]
            and DEVICE.type == "cuda"
        ):

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                features = dino.forward_features(images)

        else:

            features = dino.forward_features(images)

        # ----------------------------------------------------
        # Global CLS features
        # ----------------------------------------------------

        cls_features = features[
            "x_norm_clstoken"
        ]

        # ----------------------------------------------------
        # Patch tokens
        # ----------------------------------------------------

        patch_features = features[
            "x_norm_patchtokens"
        ]

        # ----------------------------------------------------
        # Local representation: mean of all patch tokens
        # ----------------------------------------------------

        mean_patch_features = patch_features.mean(
            dim=1
        )

        # ----------------------------------------------------
        # Combined global + local representation
        # ----------------------------------------------------

        combined = torch.cat(
            [
                cls_features,
                mean_patch_features
            ],
            dim=1
        )

        # ----------------------------------------------------
        # CPU arrays
        # ----------------------------------------------------

        global_features.append(
            cls_features.float().cpu().numpy()
        )

        local_features.append(
            mean_patch_features.float().cpu().numpy()
        )

        combined_features.append(
            combined.float().cpu().numpy()
        )

        sample_ids.extend(ids)
        image_paths.extend(paths)

    # ========================================================
    # CONCATENATE
    # ========================================================

    global_features = np.concatenate(
        global_features,
        axis=0
    ).astype(np.float32, copy=False)

    local_features = np.concatenate(
        local_features,
        axis=0
    ).astype(np.float32, copy=False)

    combined_features = np.concatenate(
        combined_features,
        axis=0
    ).astype(np.float32, copy=False)

    # ========================================================
    # SHAPE CHECKS
    # ========================================================

    assert global_features.shape == (
        len(sample_ids),
        1024
    )

    assert local_features.shape == (
        len(sample_ids),
        1024
    )

    assert combined_features.shape == (
        len(sample_ids),
        2048
    )

    assert len(image_paths) == len(sample_ids)

    # ========================================================
    # SAVE FEATURES
    # ========================================================

    split_dir = FEATURE_DIR / split_name

    split_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    save_dtype = np.float32

    if CFG_DINO["save_dtype"] == "float16":
        save_dtype = np.float16

    np.save(
        split_dir / "global_cls_1024.npy",
        global_features.astype(save_dtype)
    )

    np.save(
        split_dir / "local_mean_patch_1024.npy",
        local_features.astype(save_dtype)
    )

    np.save(
        split_dir / "global_local_2048.npy",
        combined_features.astype(save_dtype)
    )

    # --------------------------------------------------------
    # Metadata: preserve the association between feature rows,
    # original sample IDs, and augmented image paths.
    # --------------------------------------------------------

    metadata = pd.DataFrame(
        {
            "sample_id": sample_ids,
            "image_path": image_paths
        }
    )

    metadata.to_csv(
        split_dir / "metadata.csv",
        index=False
    )

    # --------------------------------------------------------
    # Feature dimensions and counts
    # --------------------------------------------------------

    np.savez(
        split_dir / "feature_info.npz",

        n_images=len(sample_ids),

        global_dim=global_features.shape[1],

        local_dim=local_features.shape[1],

        combined_dim=combined_features.shape[1]
    )

    return {
        "global": global_features,
        "local": local_features,
        "combined": combined_features,
        "sample_ids": sample_ids,
        "image_paths": image_paths
    }


# ============================================================
# EXTRACT TRAIN FEATURES
# ============================================================

print()
print("=" * 80)
print("EXTRACTING TRAIN FEATURES")
print("=" * 80)

t0 = time.time()

train_features = extract_features(
    train_loader,
    "train"
)

print(
    f"Training extraction time: "
    f"{time.time() - t0:.1f} seconds"
)


# ============================================================
# EXTRACT TEST FEATURES
# ============================================================

print()
print("=" * 80)
print("EXTRACTING TEST FEATURES")
print("=" * 80)

t0 = time.time()

test_features = extract_features(
    test_loader,
    "test"
)

print(
    f"Test extraction time: "
    f"{time.time() - t0:.1f} seconds"
)


# ============================================================
# FINAL FEATURE SHAPE VERIFICATION
# ============================================================

print()
print("=" * 80)
print("FEATURE EXTRACTION RESULTS")
print("=" * 80)

for split_name, data in [
    ("TRAIN", train_features),
    ("TEST", test_features)
]:

    print()
    print(split_name)

    print(
        "Images:",
        len(data["sample_ids"])
    )

    print(
        "Global CLS:",
        data["global"].shape
    )

    print(
        "Mean patch:",
        data["local"].shape
    )

    print(
        "Global + local:",
        data["combined"].shape
    )


# ============================================================
# ASSERT EXPECTED SHAPES
# ============================================================

assert train_features["global"].shape == (
    10160, 1024
)

assert train_features["local"].shape == (
    10160, 1024
)

assert train_features["combined"].shape == (
    10160, 2048
)

assert test_features["global"].shape == (
    35, 1024
)

assert test_features["local"].shape == (
    35, 1024
)

assert test_features["combined"].shape == (
    35, 2048
)


# ============================================================
# VERIFY FEATURES PER PHYSICAL SAMPLE
# ============================================================

print()
print("=" * 80)
print("TRAINING SAMPLE DISTRIBUTION")
print("=" * 80)

train_metadata = pd.DataFrame(
    {
        "sample_id": train_features["sample_ids"]
    }
)

train_counts = (
    train_metadata["sample_id"]
    .value_counts()
    .sort_index()
)

print(
    train_counts.to_string()
)

assert train_metadata["sample_id"].nunique() == 24

# A physical sample can have multiple original photographs.
# Each original photograph contributes exactly 80 features.
assert np.all(
    train_counts.to_numpy() % 80 == 0
)

print()
print("Unique training samples:", len(train_counts))
print("Total augmented feature rows:", train_counts.sum())
print("All sample counts are multiples of 80.")


# ============================================================
# SAVE FEATURE SUMMARY
# ============================================================

summary = pd.DataFrame(
    [
        {
            "split": "train",

            "images": len(train_features["sample_ids"]),

            "physical_samples": train_metadata[
                "sample_id"
            ].nunique(),

            "global_dim": train_features[
                "global"
            ].shape[1],

            "local_dim": train_features[
                "local"
            ].shape[1],

            "combined_dim": train_features[
                "combined"
            ].shape[1]
        },

        {
            "split": "test",

            "images": len(test_features["sample_ids"]),

            "physical_samples": len(
                set(test_features["sample_ids"])
            ),

            "global_dim": test_features[
                "global"
            ].shape[1],

            "local_dim": test_features[
                "local"
            ].shape[1],

            "combined_dim": test_features[
                "combined"
            ].shape[1]
        }
    ]
)

summary.to_csv(
    FEATURE_DIR / "feature_summary.csv",
    index=False
)

print()
print(summary.to_string(index=False))


# ============================================================
# FREE GPU MEMORY
# ============================================================

del dino
del train_loader
del test_loader

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 80)
print("DINOv2 FEATURE EXTRACTION COMPLETE")
print("=" * 80)

print("Feature directory:", FEATURE_DIR)

print()
print("TRAIN:")
print(FEATURE_DIR / "train")

print()
print("TEST:")
print(FEATURE_DIR / "test")

print()
print("Saved representations:")

print("1. global_cls_1024.npy")
print("2. local_mean_patch_1024.npy")
print("3. global_local_2048.npy")
print("4. metadata.csv")
print("5. feature_info.npz")

print()
print("Expected feature shapes:")

print("Train global       : (10160, 1024)")
print("Train local        : (10160, 1024)")
print("Train global_local : (10160, 2048)")

print("Test global        : (35, 1024)")
print("Test local         : (35, 1024)")
print("Test global_local  : (35, 2048)")

print()
print("Validation split: NONE")
print("DINO training: NONE")
print("Feature extraction: COMPLETE")
print("DONE.")


DINOv2 ViT-L/14 FEATURE EXTRACTION
Device: cuda
GPU: Tesla T4
CUDA: 12.8

Training directory: /kaggle/working/soil_augmented_dataset/train
Test directory: /kaggle/working/soil_augmented_dataset/test
Feature directory: /kaggle/working/dinov2_l_features

DISCOVERING IMAGES

Training augmented images: 10160
Test original images: 35

Unique training physical samples: 24
Unique test physical samples: 10

VERIFYING TRAINING AUGMENTATIONS
Original training photos: 127
Augmentations per photo: 80
Verified training images: 10160

LOADING DINOv2 ViT-L/14


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


DINOv2 loaded.
Load time: 5.5 seconds

VERIFYING DINO OUTPUT
Standard DINO output shape: torch.Size([16, 1024])

EXTRACTING TRAIN FEATURES


DINO train:   0%|          | 0/635 [00:00<?, ?batch/s]

Training extraction time: 376.2 seconds

EXTRACTING TEST FEATURES


DINO test:   0%|          | 0/3 [00:00<?, ?batch/s]

Test extraction time: 2.1 seconds

FEATURE EXTRACTION RESULTS

TRAIN
Images: 10160
Global CLS: (10160, 1024)
Mean patch: (10160, 1024)
Global + local: (10160, 2048)

TEST
Images: 35
Global CLS: (35, 1024)
Mean patch: (35, 1024)
Global + local: (35, 2048)

TRAINING SAMPLE DISTRIBUTION
sample_id
F827    400
G190    400
H030    560
H037    480
H038    640
H126    320
H181    480
H183    560
H366    320
H367    320
H368    400
H371    560
H372    480
H374    240
H405    480
H493    400
H516    320
H549    400
H615    400
H616    400
H617    480
H637    240
H666    320
H668    560

Unique training samples: 24
Total augmented feature rows: 10160
All sample counts are multiples of 80.

split  images  physical_samples  global_dim  local_dim  combined_dim
train   10160                24        1024       1024          2048
 test      35                10        1024       1024          2048

DINOv2 FEATURE EXTRACTION COMPLETE
Feature directory: /kaggle/working/dinov2_l_features

TRAIN:
/kaggle/

In [19]:

# ============================================================
# DINOv2 2048 + FIVE DoG FEATURES
# STANDARDIZE, CONCATENATE AND SAVE
# ============================================================

import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

FEATURE_ROOT = Path("/kaggle/working/dinov2_l_features")

TRAIN_DIR = FEATURE_ROOT / "train"
TEST_DIR = FEATURE_ROOT / "test"

# Use the existing physical calibration from your pipeline.
# CFG["target_ppm"] must be pixels per millimetre.
TARGET_PPM = float(4.6)

assert TARGET_PPM > 0, "target_ppm must be positive."

DOG_SCALES_MM = [0.5, 1.0, 2.0, 4.0, 8.0]

DOG_NAMES = [
    "dog_0.5mm",
    "dog_1.0mm",
    "dog_2.0mm",
    "dog_4.0mm",
    "dog_8.0mm",
]

assert len(DOG_SCALES_MM) == 5

print("=" * 70)
print("DINOv2 + FIVE DoG FEATURES")
print("=" * 70)
print("Feature root:", FEATURE_ROOT)
print("Target pixels/mm:", TARGET_PPM)
print("DoG scales:", DOG_SCALES_MM)


# ============================================================
# DoG FEATURE EXTRACTION
# ============================================================

def dog_features(gray: np.ndarray) -> dict:
    """
    Calculate five multiscale DoG features.

    dog_0.5mm: original image vs. 0.5 mm blur
    dog_1.0mm: 0.5 mm blur vs. 1.0 mm blur
    dog_2.0mm: 1.0 mm blur vs. 2.0 mm blur
    dog_4.0mm: 2.0 mm blur vs. 4.0 mm blur
    dog_8.0mm: 4.0 mm blur vs. 8.0 mm blur

    Each feature is the logarithm of the DoG response
    standard deviation relative to image contrast.
    """

    gray = np.asarray(gray, dtype=np.float32)

    if gray.ndim != 2:
        raise ValueError(
            f"Expected grayscale image; got shape {gray.shape}"
        )

    # Relative contrast normalization.
    image_std = float(gray.std()) + 1e-6

    # First level is the original image, avoiding an invalid
    # GaussianBlur call with sigma=0 and kernel size=(0, 0).
    blur_levels = [gray]

    for scale_mm in DOG_SCALES_MM:
        sigma_pixels = scale_mm * TARGET_PPM

        blurred = cv2.GaussianBlur(
            gray,
            (0, 0),
            sigmaX=sigma_pixels,
            sigmaY=sigma_pixels,
        )

        blur_levels.append(blurred)

    features = {}

    for k, name in enumerate(DOG_NAMES, start=1):
        dog_response = blur_levels[k - 1] - blur_levels[k]

        relative_energy = (
            float(dog_response.std()) / image_std
        )

        features[name] = float(
            np.log(relative_energy + 1e-6)
        )

    return features


# ============================================================
# COMPUTE DOG FEATURES IN METADATA ROW ORDER
# ============================================================

def extract_dog_features(split_dir: Path) -> np.ndarray:
    metadata_path = split_dir / "metadata.csv"

    if not metadata_path.exists():
        raise FileNotFoundError(metadata_path)

    metadata = pd.read_csv(metadata_path)

    if "image_path" not in metadata.columns:
        raise ValueError(
            f"'image_path' not found in {metadata_path}"
        )

    dog_rows = []

    for image_path in tqdm(
        metadata["image_path"],
        desc=f"DoG {split_dir.name}",
    ):
        image_path = Path(image_path)

        if not image_path.is_file():
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        gray = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if gray is None:
            raise ValueError(
                f"Could not read image: {image_path}"
            )

        features = dog_features(gray)

        dog_rows.append(
            [features[name] for name in DOG_NAMES]
        )

    dog_matrix = np.asarray(
        dog_rows,
        dtype=np.float32,
    )

    if dog_matrix.shape != (len(metadata), 5):
        raise ValueError(
            f"Unexpected DoG shape: {dog_matrix.shape}"
        )

    if not np.isfinite(dog_matrix).all():
        raise ValueError(
            f"Non-finite DoG values in {split_dir.name}"
        )

    return dog_matrix


# ============================================================
# LOAD EXISTING DINOv2 FEATURES
# ============================================================

X_dino_train = np.load(
    TRAIN_DIR / "global_local_2048.npy"
).astype(np.float32)

X_dino_test = np.load(
    TEST_DIR / "global_local_2048.npy"
).astype(np.float32)

train_metadata = pd.read_csv(TRAIN_DIR / "metadata.csv")
test_metadata = pd.read_csv(TEST_DIR / "metadata.csv")

assert X_dino_train.shape == (len(train_metadata), 2048)
assert X_dino_test.shape == (len(test_metadata), 2048)

print("\nDINOv2 training:", X_dino_train.shape)
print("DINOv2 test:", X_dino_test.shape)


# ============================================================
# EXTRACT FIVE DOG FEATURES PER IMAGE
# ============================================================

X_dog_train = extract_dog_features(TRAIN_DIR)
X_dog_test = extract_dog_features(TEST_DIR)

print("\nDoG training:", X_dog_train.shape)
print("DoG test:", X_dog_test.shape)


# ============================================================
# SAVE RAW DOG FEATURES
# ============================================================

np.save(TRAIN_DIR / "dog_5.npy", X_dog_train)
np.save(TEST_DIR / "dog_5.npy", X_dog_test)

dog_columns = DOG_NAMES

train_dog_df = pd.DataFrame(
    X_dog_train,
    columns=dog_columns,
)

test_dog_df = pd.DataFrame(
    X_dog_test,
    columns=dog_columns,
)

train_dog_df.insert(
    0, "image_path", train_metadata["image_path"].values
)

test_dog_df.insert(
    0, "image_path", test_metadata["image_path"].values
)

train_dog_df.to_csv(
    TRAIN_DIR / "dog_5.csv",
    index=False,
)

test_dog_df.to_csv(
    TEST_DIR / "dog_5.csv",
    index=False,
)


# ============================================================
# STANDARDIZE THE TWO FEATURE BLOCKS SEPARATELY
# ============================================================

# Fit on training data only.
dino_scaler = StandardScaler()
dog_scaler = StandardScaler()

X_dino_train_scaled = dino_scaler.fit_transform(
    X_dino_train
).astype(np.float32)

X_dog_train_scaled = dog_scaler.fit_transform(
    X_dog_train
).astype(np.float32)

# Apply the same training statistics to test data.
X_dino_test_scaled = dino_scaler.transform(
    X_dino_test
).astype(np.float32)

X_dog_test_scaled = dog_scaler.transform(
    X_dog_test
).astype(np.float32)


# ============================================================
# CONCATENATE: 2048 + 5 = 2053 FEATURES
# ============================================================

X_train_merged_raw = np.concatenate(
    [X_dino_train, X_dog_train],
    axis=1,
).astype(np.float32)

X_test_merged_raw = np.concatenate(
    [X_dino_test, X_dog_test],
    axis=1,
).astype(np.float32)

X_train_merged_scaled = np.concatenate(
    [X_dino_train_scaled, X_dog_train_scaled],
    axis=1,
).astype(np.float32)

X_test_merged_scaled = np.concatenate(
    [X_dino_test_scaled, X_dog_test_scaled],
    axis=1,
).astype(np.float32)


# ============================================================
# VALIDATE
# ============================================================

assert X_train_merged_raw.shape == (
    len(train_metadata), 2053
)

assert X_test_merged_raw.shape == (
    len(test_metadata), 2053
)

assert X_train_merged_scaled.shape == (
    len(train_metadata), 2053
)

assert X_test_merged_scaled.shape == (
    len(test_metadata), 2053
)

assert np.isfinite(X_train_merged_scaled).all()
assert np.isfinite(X_test_merged_scaled).all()


# ============================================================
# SAVE MERGED FEATURES
# ============================================================

# Raw concatenated features.
np.save(
    TRAIN_DIR / "global_local_dog_2053_raw.npy",
    X_train_merged_raw,
)

np.save(
    TEST_DIR / "global_local_dog_2053_raw.npy",
    X_test_merged_raw,
)

# Standardized concatenated features.
np.save(
    TRAIN_DIR / "global_local_dog_2053_standardized.npy",
    X_train_merged_scaled,
)

np.save(
    TEST_DIR / "global_local_dog_2053_standardized.npy",
    X_test_merged_scaled,
)

# Save scaler statistics so the same transformation can be reused.
np.savez(
    FEATURE_ROOT / "global_local_dog_scalers.npz",
    dino_mean=dino_scaler.mean_,
    dino_scale=dino_scaler.scale_,
    dog_mean=dog_scaler.mean_,
    dog_scale=dog_scaler.scale_,
    dog_names=np.asarray(DOG_NAMES),
    target_ppm=np.asarray(TARGET_PPM),
)


# ============================================================
# SAVE FEATURE INFORMATION
# ============================================================

np.savez(
    FEATURE_ROOT / "global_local_dog_feature_info.npz",
    train_shape=np.asarray(X_train_merged_scaled.shape),
    test_shape=np.asarray(X_test_merged_scaled.shape),
    dino_dim=np.asarray(2048),
    dog_dim=np.asarray(5),
    combined_dim=np.asarray(2053),
    dog_names=np.asarray(DOG_NAMES),
    target_ppm=np.asarray(TARGET_PPM),
    standardization=np.asarray(
        "Separate StandardScaler per feature block; "
        "fitted on training rows only."
    ),
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DINOv2 + DoG FEATURE MERGING COMPLETE")
print("=" * 70)

print("\nTRAIN")
print("DINOv2:", X_dino_train.shape)
print("DoG:", X_dog_train.shape)
print("Merged raw:", X_train_merged_raw.shape)
print("Merged standardized:", X_train_merged_scaled.shape)

print("\nTEST")
print("DINOv2:", X_dino_test.shape)
print("DoG:", X_dog_test.shape)
print("Merged raw:", X_test_merged_raw.shape)
print("Merged standardized:", X_test_merged_scaled.shape)

print("\nSaved files:")
for split_dir in [TRAIN_DIR, TEST_DIR]:
    for filename in [
        "dog_5.npy",
        "dog_5.csv",
        "global_local_dog_2053_raw.npy",
        "global_local_dog_2053_standardized.npy",
    ]:
        print(split_dir / filename)

print(
    FEATURE_ROOT / "global_local_dog_scalers.npz"
)
print(
    FEATURE_ROOT / "global_local_dog_feature_info.npz"
)

print("\nAll shapes and finite-value checks passed.")


DINOv2 + FIVE DoG FEATURES
Feature root: /kaggle/working/dinov2_l_features
Target pixels/mm: 4.6
DoG scales: [0.5, 1.0, 2.0, 4.0, 8.0]

DINOv2 training: (10160, 2048)
DINOv2 test: (35, 2048)


DoG train:   0%|          | 0/10160 [00:00<?, ?it/s]

DoG test:   0%|          | 0/35 [00:00<?, ?it/s]


DoG training: (10160, 5)
DoG test: (35, 5)

DINOv2 + DoG FEATURE MERGING COMPLETE

TRAIN
DINOv2: (10160, 2048)
DoG: (10160, 5)
Merged raw: (10160, 2053)
Merged standardized: (10160, 2053)

TEST
DINOv2: (35, 2048)
DoG: (35, 5)
Merged raw: (35, 2053)
Merged standardized: (35, 2053)

Saved files:
/kaggle/working/dinov2_l_features/train/dog_5.npy
/kaggle/working/dinov2_l_features/train/dog_5.csv
/kaggle/working/dinov2_l_features/train/global_local_dog_2053_raw.npy
/kaggle/working/dinov2_l_features/train/global_local_dog_2053_standardized.npy
/kaggle/working/dinov2_l_features/test/dog_5.npy
/kaggle/working/dinov2_l_features/test/dog_5.csv
/kaggle/working/dinov2_l_features/test/global_local_dog_2053_raw.npy
/kaggle/working/dinov2_l_features/test/global_local_dog_2053_standardized.npy
/kaggle/working/dinov2_l_features/global_local_dog_scalers.npz
/kaggle/working/dinov2_l_features/global_local_dog_feature_info.npz

All shapes and finite-value checks passed.


In [25]:
# DINOv2 ViT-L/14 + 5 DoG (2053 features)
# End-to-end flexible-Weibull CV + final submission.
# Run in Kaggle after feature extraction has produced:
#   train/metadata.csv, train/global_local_dog_2053_raw.npy
#   test/metadata.csv,  test/global_local_dog_2053_raw.npy
#
# Important fixes:
# - New output directory prevents stale files from mixing across runs.
# - LOO explicitly creates one fold for every labelled physical sample.
# - CV scaler is fitted on training rows only; final train/test share one scaler.
# - Sample-balanced WeightedRandomSampler reduces augmentation-count imbalance.
# - Fold failures are logged and all fold results are saved incrementally.
# - Final epoch count comes from median best epoch across successful LOO folds.
# - Final submission is stopped if any LOO sample is missing.

import gc, re, time, random, warnings, traceback
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

CFG = {
    "feature_dir": "/kaggle/working/dinov2_l_features",
    "output_dir": "/kaggle/working/dinov2_dog5_psd_outputs_v2",
    "feature_dim": 2053, "batch_size": 128, "epochs": 100,
    "learning_rate": 3.1e-6, "weight_decay": 1e-2, "dropout": 0.40,
    "mse_weight": 0.10, "emd_weight": 0.90, "emd_normalizer": 100.0,
    "fine_emd_weights": [1.0] * 10,  # Keep unweighted until full CV is verified.
    "patience": 8, "min_emd_improvement": 1e-4,
    "scheduler_factor": 0.5, "scheduler_patience": 3,
    "scheduler_min_lr": 1e-7, "num_workers": 0,
    "minimum_train_rows": 80, "minimum_train_samples": 3,
    "show_fold_progress": True, "save_fold_checkpoints": True,
}
FEATURE_DIR = Path(CFG["feature_dir"])
OUT = Path(CFG["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FD = CFG["feature_dim"]

print("=" * 80)
print("DINOv2 + 5 DoG FLEXIBLE WEIBULL PIPELINE")
print("Device:", DEVICE, "| Features:", FD, "| Output:", OUT)
print("=" * 80)

# ---------- Labels / submission ----------
inputs = Path("/kaggle/input")
label_files = sorted(inputs.glob("**/Training_labels_updated.csv"))
sub_files = sorted(inputs.glob("**/sample_submission.csv"))
if not label_files: raise FileNotFoundError("Training_labels_updated.csv not found.")
if not sub_files: raise FileNotFoundError("sample_submission.csv not found.")
labels_raw = pd.read_csv(label_files[0], dtype={"sample_id": str})
submission_template = pd.read_csv(sub_files[0], dtype={"sample_id": str})
if "sample_id" not in labels_raw or "sample_id" not in submission_template:
    raise ValueError("Labels and submission template must contain sample_id.")
labels_raw["sample_id"] = labels_raw["sample_id"].astype(str).str.strip()
submission_template["sample_id"] = submission_template["sample_id"].astype(str).str.strip()
if labels_raw.sample_id.duplicated().any(): raise ValueError("Duplicate sample_id in labels.")
labels = labels_raw.set_index("sample_id")
SIZES = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2., 6.3, 20., 63., 200.])
SIZE_COLS = ["0.002","0.0063","0.02","0.063","0.2","0.63","2","6.3","20","63","200"]
missing = [c for c in SIZE_COLS if c not in labels.columns]
if missing: raise ValueError(f"Missing target columns: {missing}")
labels[SIZE_COLS] = labels[SIZE_COLS].apply(pd.to_numeric, errors="raise")
if labels[SIZE_COLS].isna().any().any(): raise ValueError("Missing target values.")
LABEL_IDS = set(labels.index.astype(str))
if len(LABEL_IDS) != 24:
    print(f"WARNING: expected 24 labels based on prior audit; found {len(LABEL_IDS)}.")

LOG_SIZES = np.log(SIZES)
W_EMD = np.diff(np.log10(SIZES))
C_FLOOR = 0.02
CENTER1, CENTER2 = float(np.log(0.02)), float(np.log(0.2))

# ---------- Flexible Weibull / metric ----------
def emd(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return (np.abs(y[..., :10] - p[..., :10]) * W_EMD).sum(axis=-1)

def postprocess(curves):
    curves = np.clip(np.asarray(curves, float).copy(), 0., 100.)
    curves = np.maximum.accumulate(curves, axis=-1)
    curves[..., -1] = 100.
    return curves

def weibull_curve(lnb, c, q1=0., q2=0.):
    lnb, c, q1, q2 = [np.asarray(v, dtype=float) for v in (lnb,c,q1,q2)]
    c = np.maximum(c, np.abs(q1) + np.abs(q2) + C_FLOOR)
    z = (c[...,None]*(LOG_SIZES-lnb[...,None])
         + q1[...,None]*np.tanh(LOG_SIZES-CENTER1)
         + q2[...,None]*np.tanh(LOG_SIZES-CENTER2))
    return 100. * (1. - np.exp(-np.exp(np.clip(z, -50., 50.))))

def fit_weibull(y):
    y = np.asarray(y, float)
    gl = np.linspace(-6., 4., 41); gc = np.linspace(.1, 2., 39)
    L, C = np.meshgrid(gl, gc, indexing="ij")
    grid = weibull_curve(L.ravel(), C.ravel(), np.zeros(L.size), np.zeros(L.size))
    err = emd(y, grid).reshape(L.shape)
    ix = np.unravel_index(np.argmin(err), err.shape)
    x0 = np.array([L[ix], max(C[ix], C_FLOOR), 0., 0.])
    def objective(x):
        v = emd(y, weibull_curve(x[0], x[1], x[2], x[3]))
        return float(v) if np.isfinite(v) else 1e12
    cons = [{"type":"ineq","fun":lambda x: x[1]-abs(x[2])-abs(x[3])-C_FLOOR}]
    res = minimize(objective, x0, method="SLSQP",
                   bounds=[(-8,5),(C_FLOOR,10),(-5,5),(-5,5)],
                   constraints=cons, options={"ftol":1e-8,"maxiter":1500})
    p = np.asarray(res.x if res.success and np.isfinite(res.fun) and res.fun <= err[ix] else x0, float)
    p[1] = max(p[1], abs(p[2])+abs(p[3])+C_FLOOR)
    return p.astype(np.float32)

# ---------- Metadata / raw features ----------
def load_meta(split):
    p = FEATURE_DIR / split / "metadata.csv"
    if not p.exists(): raise FileNotFoundError(p)
    d = pd.read_csv(p)
    if not {"sample_id","image_path"}.issubset(d.columns):
        raise ValueError(f"{p} needs sample_id and image_path.")
    d["sample_id"] = d.sample_id.astype(str).str.strip()
    d["image_path"] = d.image_path.astype(str)
    return d

train_meta, test_meta = load_meta("train"), load_meta("test")

def load_raw(split, n):
    p = FEATURE_DIR / split / "global_local_dog_2053_raw.npy"
    if not p.exists(): raise FileNotFoundError(p)
    x = np.load(p).astype(np.float32)
    if x.shape != (n, FD): raise ValueError(f"{split} feature shape {x.shape}; expected {(n,FD)}.")
    if not np.isfinite(x).all(): raise ValueError(f"Non-finite features in {split}.")
    return x

X_RAW = load_raw("train", len(train_meta))
X_TEST_RAW = load_raw("test", len(test_meta))

def get_photo_id(row, is_test=False):
    if "photo_id" in row.index and pd.notna(row["photo_id"]):
        return str(row["photo_id"])
    p = str(row["image_path"]).replace("\\","/")
    return Path(p).stem if is_test else Path(p).parent.name

train_meta["__photo"] = train_meta.apply(lambda r: get_photo_id(r), axis=1)
test_meta["__photo"] = test_meta.apply(lambda r: get_photo_id(r, True), axis=1)

def infer_camera(photo, sid):
    photo, sid = str(photo).strip(), str(sid).strip()
    # Prefer removing sample-ID suffixes such as _F827_01 or _F827.
    for pat in [rf"_{re.escape(sid)}_\d+$", rf"_{re.escape(sid)}(?:_\d+)?$"]:
        v = re.sub(pat, "", photo, flags=re.I)
        if v != photo:
            photo = v
            break
    camera = photo.strip("_ -")
    if not camera:
        raise ValueError(f"Could not infer camera from photo={photo!r}, sample={sid!r}.")
    if re.fullmatch(r"iPhone_?16", camera, flags=re.I): camera = "iPhone16"
    return camera

for meta in (train_meta, test_meta):
    if "camera" in meta.columns:
        meta["__camera"] = meta["camera"].astype(str).str.strip()
    else:
        meta["__camera"] = [infer_camera(p,s) for p,s in zip(meta["__photo"],meta["sample_id"])]
    meta["__sample"] = meta["sample_id"].astype(str)

SAMPLE_IDS = train_meta["__sample"].to_numpy(str)
PHOTO_IDS = train_meta["__photo"].to_numpy(str)
CAMERAS = train_meta["__camera"].to_numpy(str)
TEST_SAMPLE_IDS = test_meta["__sample"].to_numpy(str)

if set(SAMPLE_IDS) != LABEL_IDS:
    raise ValueError(
        "Training feature sample IDs do not match labels. "
        f"Only features={sorted(set(SAMPLE_IDS)-LABEL_IDS)}; "
        f"only labels={sorted(LABEL_IDS-set(SAMPLE_IDS))}"
    )
for col in ("__sample","__camera"):
    nunique = train_meta.groupby("__photo")[col].nunique()
    if (nunique > 1).any(): raise ValueError(f"A photo maps to multiple {col} values.")
print("\nTraining rows:",len(train_meta),"| Samples:",len(set(SAMPLE_IDS)),
      "| Photos:",train_meta["__photo"].nunique())
print("Camera counts:\n",train_meta["__camera"].value_counts().to_string())
train_meta[["__sample","__photo","__camera"]].drop_duplicates().to_csv(OUT/"training_photo_camera_mapping.csv",index=False)
test_meta[["__sample","__photo","__camera"]].to_csv(OUT/"test_photo_camera_mapping.csv",index=False)

# ---------- One Weibull target per physical sample ----------
params_by_sample, psd_by_sample = {}, {}
fit_rows = []
for sid in tqdm(sorted(LABEL_IDS), desc="Fit Weibull labels"):
    y = labels.loc[sid, SIZE_COLS].to_numpy(float)
    p = fit_weibull(y)
    params_by_sample[sid], psd_by_sample[sid] = p, y
    fit_curve = postprocess(weibull_curve(p[0],p[1],p[2],p[3]))
    fit_rows.append({"sample_id":sid,"lnb":p[0],"c":p[1],"q1":p[2],"q2":p[3],
                     "fit_EMD":float(emd(y,fit_curve)),
                     "fit_MAE":float(np.mean(np.abs(y-fit_curve)))})
pd.DataFrame(fit_rows).to_csv(OUT/"fitted_weibull_targets.csv",index=False)
Y_PARAM = np.stack([params_by_sample[s] for s in SAMPLE_IDS]).astype(np.float32)
Y_PSD = np.stack([psd_by_sample[s] for s in SAMPLE_IDS]).astype(np.float32)
print("Mean Weibull fitting EMD:", round(pd.DataFrame(fit_rows).fit_EMD.mean(),4))

# ---------- Torch model ----------
SIZES_T = torch.tensor(SIZES,dtype=torch.float32,device=DEVICE)
W_EMD_T = torch.tensor(W_EMD,dtype=torch.float32,device=DEVICE)
FINE_W_T = torch.tensor(CFG["fine_emd_weights"],dtype=torch.float32,device=DEVICE)
if len(CFG["fine_emd_weights"]) != 10: raise ValueError("fine_emd_weights needs 10 entries.")

def weibull_torch(p):
    lnb,c,q1,q2 = p[:,0:1],p[:,1:2],p[:,2:3],p[:,3:4]
    c = torch.maximum(c,torch.abs(q1)+torch.abs(q2)+C_FLOOR)
    ls = torch.log(SIZES_T).view(1,-1)
    z = c*(ls-lnb)+q1*torch.tanh(ls-CENTER1)+q2*torch.tanh(ls-CENTER2)
    return 100.*(1.-torch.exp(-torch.exp(torch.clamp(z,-50.,50.))))

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FD,32),nn.LayerNorm(32),nn.ReLU(),nn.Dropout(CFG["dropout"]),
            nn.Linear(32,32),nn.LayerNorm(32),nn.ReLU(),nn.Dropout(CFG["dropout"]),
            nn.Linear(32,32),nn.LayerNorm(32),nn.ReLU(),nn.Dropout(CFG["dropout"]),
            nn.Linear(32,4))
    def forward(self,x): return self.net(x)

class DS(Dataset):
    def __init__(self,x,yp,ys,sids):
        self.x=np.asarray(x,np.float32); self.yp=np.asarray(yp,np.float32)
        self.ys=np.asarray(ys,np.float32); self.sids=np.asarray(sids,str)
    def __len__(self): return len(self.x)
    def __getitem__(self,i):
        return (torch.from_numpy(self.x[i]),torch.from_numpy(self.yp[i]),
                torch.from_numpy(self.ys[i]),self.sids[i])

def train_loader(x,yp,ys,sids):
    counts=Counter(np.asarray(sids,str).tolist())
    weights=torch.as_tensor([1./counts[s] for s in sids],dtype=torch.double)
    sampler=WeightedRandomSampler(weights,len(weights),replacement=True)
    return DataLoader(DS(x,yp,ys,sids),batch_size=CFG["batch_size"],sampler=sampler,
                      num_workers=CFG["num_workers"],pin_memory=DEVICE.type=="cuda")

def eval_loader(x,yp,ys,sids):
    return DataLoader(DS(x,yp,ys,sids),batch_size=CFG["batch_size"],shuffle=False,
                      num_workers=CFG["num_workers"],pin_memory=DEVICE.type=="cuda")

def loss_fn(pred_std, ystd, ypsd, mean_t, scale_t):
    mse=nn.functional.mse_loss(pred_std,ystd)
    params=pred_std*scale_t.view(1,-1)+mean_t.view(1,-1)
    curve=weibull_torch(params)
    curve_loss=(torch.abs(curve[:,:10]-ypsd[:,:10])*W_EMD_T.view(1,-1)*FINE_W_T.view(1,-1)).sum(1).mean()
    total=CFG["mse_weight"]*mse+CFG["emd_weight"]*curve_loss/CFG["emd_normalizer"]
    return total,mse,curve_loss

@torch.no_grad()
def predict(model,loader,mean,scale):
    model.eval(); ps=[]; ids=[]
    for x,_,_,sids in loader:
        ps.append(model(x.to(DEVICE)).cpu().numpy()); ids.extend(list(sids))
    if not ps: raise RuntimeError("Empty prediction loader.")
    z=np.concatenate(ps)
    params=z*np.asarray(scale)[None,:]+np.asarray(mean)[None,:]
    curves=weibull_curve(params[:,0],params[:,1],params[:,2],params[:,3])
    return params,curves,np.asarray(ids,str)

def aggregate(curves,idx):
    f=pd.DataFrame(curves,columns=SIZE_COLS)
    f["sample_id"]=SAMPLE_IDS[idx]; f["photo_id"]=PHOTO_IDS[idx]
    photo=f.groupby(["sample_id","photo_id"],sort=False)[SIZE_COLS].mean()
    sample=photo.groupby(level="sample_id",sort=False)[SIZE_COLS].mean()
    sample.loc[:,:]=postprocess(sample.to_numpy())
    scores={s:float(emd(psd_by_sample[s],sample.loc[s,SIZE_COLS].to_numpy())) for s in sample.index.astype(str)}
    return sample,photo,scores,float(np.mean(list(scores.values())))

def run_fold(experiment,name,tr_idx,va_idx):
    tr_idx=np.asarray(tr_idx,int); va_idx=np.asarray(va_idx,int)
    tr_samples=sorted(set(SAMPLE_IDS[tr_idx])); va_samples=sorted(set(SAMPLE_IDS[va_idx]))
    overlap=set(tr_samples)&set(va_samples)
    if overlap: raise ValueError(f"Physical-sample leakage: {sorted(overlap)}")
    if not len(tr_idx) or not len(va_idx): raise ValueError("Empty fold.")

    fs=StandardScaler()
    xt=fs.fit_transform(X_RAW[tr_idx]).astype(np.float32)
    xv=fs.transform(X_RAW[va_idx]).astype(np.float32)
    ts=StandardScaler().fit(np.stack([params_by_sample[s] for s in tr_samples]))
    yt=ts.transform(Y_PARAM[tr_idx]).astype(np.float32)
    yv=ts.transform(Y_PARAM[va_idx]).astype(np.float32)
    tl=train_loader(xt,yt,Y_PSD[tr_idx],SAMPLE_IDS[tr_idx])
    vl=eval_loader(xv,yv,Y_PSD[va_idx],SAMPLE_IDS[va_idx])

    model=Net().to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=CFG["learning_rate"],weight_decay=CFG["weight_decay"])
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,mode="min",factor=CFG["scheduler_factor"],patience=CFG["scheduler_patience"],
        threshold=CFG["min_emd_improvement"],threshold_mode="abs",min_lr=CFG["scheduler_min_lr"])
    mean_t=torch.tensor(ts.mean_,dtype=torch.float32,device=DEVICE)
    scale_t=torch.tensor(ts.scale_,dtype=torch.float32,device=DEVICE)
    ckpt=OUT/f"best_{experiment}_{name}.pth"
    hist_path=OUT/f"history_{experiment}_{name}.csv"
    hist=[]; best=float("inf"); best_epoch=0; bad=0

    print(f"\n{experiment} | {name} | train samples={len(tr_samples)} rows={len(tr_idx)} "
          f"| val samples={len(va_samples)} rows={len(va_idx)}")
    for epoch in range(1,CFG["epochs"]+1):
        t0=time.time(); model.train()
        ls=ms=cs=0.; nseen=0
        for x,yp,ys,_ in tl:
            x=x.to(DEVICE); yp=yp.to(DEVICE); ys=ys.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pred=model(x); loss,mse,cl=loss_fn(pred,yp,ys,mean_t,scale_t)
            loss.backward(); opt.step()
            n=len(x); ls+=loss.item()*n; ms+=mse.item()*n; cs+=cl.item()*n; nseen+=n
        _,vc,_=predict(model,vl,ts.mean_,ts.scale_)
        _,_,_,ve=aggregate(vc,va_idx); sched.step(ve)
        row={"experiment":experiment,"fold":name,"epoch":epoch,
             "train_loss":ls/max(nseen,1),"train_parameter_loss":ms/max(nseen,1),
             "train_curve_loss":cs/max(nseen,1),"val_sample_emd":ve,
             "learning_rate":opt.param_groups[0]["lr"],"seconds":time.time()-t0}
        hist.append(row); improved=ve<best-CFG["min_emd_improvement"]
        if improved:
            best=ve; best_epoch=epoch; bad=0
            if CFG["save_fold_checkpoints"]:
                torch.save({"model_state_dict":model.state_dict(),"epoch":epoch,
                    "best_val_sample_emd":best,"experiment":experiment,"fold":name,
                    "feature_dim":FD,"feature_scaler_mean":fs.mean_,"feature_scaler_scale":fs.scale_,
                    "target_scaler_mean":ts.mean_,"target_scaler_scale":ts.scale_,
                    "train_sample_ids":tr_samples,"val_sample_ids":va_samples},ckpt)
        else: bad+=1
        pd.DataFrame(hist).to_csv(hist_path,index=False)
        if epoch==1 or epoch%5==0 or improved:
            print(f"Epoch {epoch:03d} train={row['train_loss']:.5f} valEMD={ve:.4f} "
                  f"lr={row['learning_rate']:.2e}{' BEST' if improved else ''}")
        if bad>=CFG["patience"]:
            print(f"Early stop at {epoch}; best epoch {best_epoch}, EMD {best:.4f}")
            break
    if best_epoch==0: raise RuntimeError("No best checkpoint selected.")
    if CFG["save_fold_checkpoints"]:
        try: state=torch.load(ckpt,map_location=DEVICE,weights_only=False)
        except TypeError: state=torch.load(ckpt,map_location=DEVICE)
        model.load_state_dict(state["model_state_dict"])
    _,best_curves,_=predict(model,vl,ts.mean_,ts.scale_)
    sample,photo,scores,final_score=aggregate(best_curves,va_idx)
    sample.to_csv(OUT/f"val_sample_psd_{experiment}_{name}.csv")
    photo.to_csv(OUT/f"val_photo_psd_{experiment}_{name}.csv")
    pd.DataFrame([{"sample_id":s,"EMD":v} for s,v in scores.items()]).to_csv(
        OUT/f"val_scores_{experiment}_{name}.csv",index=False)
    result={"experiment":experiment,"fold":name,"held_out_samples":"|".join(va_samples),
            "training_cameras":"|".join(sorted(set(CAMERAS[tr_idx]))),
            "held_out_cameras":"|".join(sorted(set(CAMERAS[va_idx]))),
            "train_rows":len(tr_idx),"val_rows":len(va_idx),"train_samples":len(tr_samples),
            "val_samples":len(va_samples),"best_epoch":best_epoch,
            "best_sample_emd":final_score,"checkpoint":str(ckpt),"status":"success","error":""}
    del model,opt,sched,tl,vl
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

# ---------- Build exact LOO folds ----------
def safe_name(s): return re.sub(r"[^A-Za-z0-9_.-]+","_",str(s))[:100]

loo_folds=[]
for sid in sorted(LABEL_IDS):
    vi=np.flatnonzero(SAMPLE_IDS==sid); ti=np.flatnonzero(SAMPLE_IDS!=sid)
    if not len(vi): raise ValueError(f"No feature rows for LOO sample {sid}.")
    if sid in set(SAMPLE_IDS[ti]): raise RuntimeError(f"LOO leakage for {sid}.")
    if len(ti)<CFG["minimum_train_rows"] or len(set(SAMPLE_IDS[ti]))<CFG["minimum_train_samples"]:
        raise ValueError(f"Insufficient training rows/samples for LOO {sid}.")
    loo_folds.append((f"sample_{safe_name(sid)}",ti,vi))
held=[SAMPLE_IDS[vi[0]] for _,_,vi in loo_folds]
if len(loo_folds)!=len(LABEL_IDS) or set(held)!=LABEL_IDS:
    raise RuntimeError("LOO construction failed to cover every labelled sample.")
pd.DataFrame([{"fold":n,"heldout_sample":SAMPLE_IDS[vi[0]],
               "train_rows":len(ti),"val_rows":len(vi),
               "train_samples":len(set(SAMPLE_IDS[ti]))}
              for n,ti,vi in loo_folds]).to_csv(OUT/"planned_sample_loo_folds.csv",index=False)

# ---------- Build cross-phone folds ----------
cross_folds=[]
sample_cameras=train_meta.groupby("__sample")["__camera"].apply(lambda x:set(x.astype(str))).to_dict()
camera_names=sorted(train_meta["__camera"].astype(str).unique())
for sid in sorted(sample_cameras):
    for target_cam in sorted(sample_cameras[sid]):
        vi=np.flatnonzero((SAMPLE_IDS==sid)&(CAMERAS==target_cam))
        if not len(vi): continue
        for source_cam in camera_names:
            if source_cam==target_cam: continue
            ti=np.flatnonzero((CAMERAS==source_cam)&(SAMPLE_IDS!=sid))
            if len(ti)<CFG["minimum_train_rows"] or len(set(SAMPLE_IDS[ti]))<CFG["minimum_train_samples"]:
                continue
            name=f"heldout_{safe_name(sid)}__source_{safe_name(source_cam)}__target_{safe_name(target_cam)}"
            cross_folds.append((name,ti,vi))
unique={}
for n,ti,vi in cross_folds: unique[(tuple(ti),tuple(vi))]=(n,ti,vi)
cross_folds=list(unique.values())

print("\nPlanned LOO folds:",len(loo_folds)," (must equal",len(LABEL_IDS),")")
print("Planned cross-phone folds:",len(cross_folds))
pd.DataFrame([{"fold":n,"heldout_samples":"|".join(sorted(set(SAMPLE_IDS[vi]))),
               "source_cameras":"|".join(sorted(set(CAMERAS[ti]))),
               "target_cameras":"|".join(sorted(set(CAMERAS[vi])))}
              for n,ti,vi in cross_folds]).to_csv(OUT/"planned_cross_phone_folds.csv",index=False)

# ---------- Run all folds, saving after every fold ----------
results=[]; failures=[]
all_folds=[("cross_phone",cross_folds),("sample_leave_one_out",loo_folds)]
total=sum(len(fs) for _,fs in all_folds)
with tqdm(total=total,desc="All CV folds",disable=not CFG["show_fold_progress"]) as bar:
    for exp,folds in all_folds:
        for name,ti,vi in folds:
            try:
                r=run_fold(exp,name,ti,vi)
            except Exception as e:
                err=f"{type(e).__name__}: {e}"
                print(f"\nFOLD FAILED: {exp}/{name}: {err}")
                traceback.print_exc()
                r={"experiment":exp,"fold":name,"held_out_samples":"|".join(sorted(set(SAMPLE_IDS[vi]))),
                   "training_cameras":"|".join(sorted(set(CAMERAS[ti]))),
                   "held_out_cameras":"|".join(sorted(set(CAMERAS[vi]))),
                   "train_rows":len(ti),"val_rows":len(vi),
                   "train_samples":len(set(SAMPLE_IDS[ti])),"val_samples":len(set(SAMPLE_IDS[vi])),
                   "best_epoch":np.nan,"best_sample_emd":np.nan,"checkpoint":"",
                   "status":"failed","error":err}
                failures.append({"experiment":exp,"fold":name,"error":err,
                                 "traceback":traceback.format_exc()})
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            results.append(r)
            pd.DataFrame(results).to_csv(OUT/"cv_fold_results_partial.csv",index=False)
            if failures: pd.DataFrame(failures).to_csv(OUT/"cv_fold_failures.csv",index=False)
            bar.update(1)

results_df=pd.DataFrame(results)
results_df.to_csv(OUT/"cv_fold_results.csv",index=False)
if failures: pd.DataFrame(failures).to_csv(OUT/"cv_fold_failures.csv",index=False)

# ---------- Hard coverage verification ----------
succ=results_df[results_df.status=="success"].copy()
loo_ok=succ[succ.experiment=="sample_leave_one_out"].copy()
cross_ok=succ[succ.experiment=="cross_phone"].copy()
observed=set()
for v in loo_ok.held_out_samples.astype(str):
    observed.update(v.split("|"))
missing_loo=sorted(LABEL_IDS-observed)
extra_loo=sorted(observed-LABEL_IDS)
coverage=pd.DataFrame([{
    "labelled_samples":len(LABEL_IDS),"planned_loo_folds":len(loo_folds),
    "successful_loo_folds":len(loo_ok),"unique_successful_loo_samples":len(observed),
    "missing_loo_samples":"|".join(missing_loo),"extra_loo_samples":"|".join(extra_loo),
    "planned_cross_phone_folds":len(cross_folds),"successful_cross_phone_folds":len(cross_ok),
    "failed_folds":len(failures)}])
coverage.to_csv(OUT/"validation_coverage_report.csv",index=False)
print("\nVALIDATION COVERAGE\n",coverage.to_string(index=False))

summaries=[]
for exp,df in [("sample_leave_one_out",loo_ok),("cross_phone",cross_ok)]:
    if len(df):
        summaries.append({"experiment":exp,"successful_folds":len(df),
            "mean_EMD":df.best_sample_emd.mean(),"median_EMD":df.best_sample_emd.median(),
            "worst_EMD":df.best_sample_emd.max(),"median_best_epoch":df.best_epoch.median(),
            "mean_best_epoch":df.best_epoch.mean()})
summary=pd.DataFrame(summaries)
summary.to_csv(OUT/"cv_summary.csv",index=False)
print("\nCV SUMMARY\n",summary.to_string(index=False) if len(summary) else "No successful folds.")

# Do not train final model if any LOO sample is missing or any LOO fold failed.
if missing_loo:
    raise RuntimeError(
        f"LOO incomplete; missing {missing_loo}. Failure details saved in "
        f"{OUT/'cv_fold_failures.csv'}. Fix these folds and rerun before submission."
    )
if len(loo_ok)!=len(LABEL_IDS):
    raise RuntimeError("Successful LOO fold count does not equal labelled sample count.")
final_epochs=max(1,min(CFG["epochs"],int(round(loo_ok.best_epoch.median()))))
print("\nFinal epoch count (median best LOO epoch):",final_epochs)

# ---------- Final model: shared train/test feature scaler ----------
feature_scaler=StandardScaler()
X_FINAL=feature_scaler.fit_transform(X_RAW).astype(np.float32)
X_TEST=feature_scaler.transform(X_TEST_RAW).astype(np.float32)
unique_params=np.stack([params_by_sample[s] for s in sorted(LABEL_IDS)]).astype(np.float32)
target_scaler=StandardScaler().fit(unique_params)
Y_PARAM_STD=target_scaler.transform(Y_PARAM).astype(np.float32)
tl=train_loader(X_FINAL,Y_PARAM_STD,Y_PSD,SAMPLE_IDS)
model=Net().to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=CFG["learning_rate"],weight_decay=CFG["weight_decay"])
mean_t=torch.tensor(target_scaler.mean_,dtype=torch.float32,device=DEVICE)
scale_t=torch.tensor(target_scaler.scale_,dtype=torch.float32,device=DEVICE)
final_hist=[]
for epoch in tqdm(range(1,final_epochs+1),desc="Final training"):
    model.train(); ls=ms=cs=0.; nseen=0
    for x,yp,ys,_ in tl:
        x=x.to(DEVICE); yp=yp.to(DEVICE); ys=ys.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        pred=model(x); loss,mse,cl=loss_fn(pred,yp,ys,mean_t,scale_t)
        loss.backward(); opt.step()
        n=len(x); ls+=loss.item()*n; ms+=mse.item()*n; cs+=cl.item()*n; nseen+=n
    row={"epoch":epoch,"train_loss":ls/max(nseen,1),
         "train_parameter_loss":ms/max(nseen,1),"train_curve_loss":cs/max(nseen,1)}
    final_hist.append(row)
    if epoch==1 or epoch%5==0 or epoch==final_epochs:
        print(f"Epoch {epoch}/{final_epochs}: loss={row['train_loss']:.5f}, curve={row['train_curve_loss']:.3f}")
pd.DataFrame(final_hist).to_csv(OUT/"final_training_history.csv",index=False)

checkpoint_path=OUT/"final_model_dinov2_dog5.pth"
torch.save({"model_state_dict":model.state_dict(),"feature_dim":FD,
            "feature_scaler_mean":feature_scaler.mean_,"feature_scaler_scale":feature_scaler.scale_,
            "target_scaler_mean":target_scaler.mean_,"target_scaler_scale":target_scaler.scale_,
            "selected_epochs":final_epochs,"config":CFG,
            "dog_names":["dog_0.5mm","dog_1.0mm","dog_2.0mm","dog_4.0mm","dog_8.0mm"]},
           checkpoint_path)

# ---------- Test inference ----------
test_ds=DS(X_TEST,np.zeros((len(X_TEST),4),np.float32),np.zeros((len(X_TEST),11),np.float32),TEST_SAMPLE_IDS)
test_dl=DataLoader(test_ds,batch_size=CFG["batch_size"],shuffle=False,num_workers=CFG["num_workers"])
test_params=[]; test_ids=[]
model.eval()
with torch.no_grad():
    for x,_,_,ids in tqdm(test_dl,desc="Test inference"):
        test_params.append(model(x.to(DEVICE)).cpu().numpy()); test_ids.extend(list(ids))
test_z=np.concatenate(test_params)
test_p=test_z*target_scaler.scale_[None,:]+target_scaler.mean_[None,:]
test_curves=weibull_curve(test_p[:,0],test_p[:,1],test_p[:,2],test_p[:,3])
pd.DataFrame({"sample_id":test_ids,"lnb":test_p[:,0],"c":test_p[:,1],"q1":test_p[:,2],"q2":test_p[:,3]}).to_csv(
    OUT/"test_flexible_weibull_parameters.csv",index=False)

tf=pd.DataFrame(test_curves,columns=SIZE_COLS)
tf["sample_id"]=np.asarray(test_ids,str)
tf["photo_id"]=test_meta["__photo"].to_numpy()
tp=tf.groupby(["sample_id","photo_id"],sort=False)[SIZE_COLS].mean()
ts=tp.groupby(level="sample_id",sort=False)[SIZE_COLS].mean()
ts.loc[:,:]=postprocess(ts.to_numpy())
ts.to_csv(OUT/"test_sample_predictions.csv")

def canon(x): return " ".join(str(x).strip().replace("_"," ").split())
lookup={str(s).strip():s for s in ts.index.astype(str)}
clookup={canon(s):s for s in ts.index.astype(str)}
ordered=[]; missing_sub=[]
for sid in submission_template.sample_id.astype(str).str.strip():
    actual=lookup.get(sid,clookup.get(canon(sid)))
    if actual is None: missing_sub.append(sid)
    else: ordered.append(ts.loc[actual,SIZE_COLS].to_numpy(float))
if missing_sub:
    raise ValueError(f"Missing submission predictions: {missing_sub}; available IDs={list(ts.index.astype(str))}")
vals=postprocess(np.asarray(ordered,float))
sub=submission_template[["sample_id"]].copy()
sub[SIZE_COLS]=vals
assert len(sub)==len(submission_template)
assert sub.sample_id.tolist()==submission_template.sample_id.tolist()
assert np.isfinite(sub[SIZE_COLS].to_numpy()).all()
assert np.all(np.diff(sub[SIZE_COLS].to_numpy(),axis=1)>=-1e-8)
assert np.allclose(sub[SIZE_COLS].to_numpy()[:,-1],100.)
sub_path=OUT/"submission_dinov2_dog5.csv"
sub.to_csv(sub_path,index=False)

print("\n"+"="*80)
print("PIPELINE COMPLETE")
print("Successful LOO folds:",len(loo_ok),"/",len(LABEL_IDS))
print("Successful cross-phone folds:",len(cross_ok),"/",len(cross_folds))
print("CV results:",OUT/"cv_fold_results.csv")
print("CV summary:",OUT/"cv_summary.csv")
print("Coverage report:",OUT/"validation_coverage_report.csv")
print("Final checkpoint:",checkpoint_path)
print("Submission:",sub_path)
print(sub.round(3).to_string(index=False))


DINOv2 + 5 DoG FLEXIBLE WEIBULL PIPELINE
Device: cuda | Features: 2053 | Output: /kaggle/working/dinov2_dog5_psd_outputs_v2

Training rows: 10160 | Samples: 24 | Photos: 127
Camera counts:
 __camera
Motorola_Edge              5520
Samsung_A52                4400
Motorola_Edge_60_fusion     240


Fit Weibull labels:   0%|          | 0/24 [00:00<?, ?it/s]

Mean Weibull fitting EMD: 3.9725

Planned LOO folds: 24  (must equal 24 )
Planned cross-phone folds: 46


All CV folds:   0%|          | 0/70 [00:00<?, ?it/s]


cross_phone | heldout_F827__source_Samsung_A52__target_Motorola_Edge | train samples=21 rows=4160 | val samples=1 rows=160
Epoch 001 train=0.86305 valEMD=133.0767 lr=3.10e-06 BEST
Epoch 002 train=0.84668 valEMD=129.7024 lr=3.10e-06 BEST
Epoch 003 train=0.83344 valEMD=127.1435 lr=3.10e-06 BEST
Epoch 004 train=0.80199 valEMD=124.7485 lr=3.10e-06 BEST
Epoch 005 train=0.78862 valEMD=122.9904 lr=3.10e-06 BEST
Epoch 006 train=0.76935 valEMD=121.5428 lr=3.10e-06 BEST
Epoch 007 train=0.77212 valEMD=120.3414 lr=3.10e-06 BEST
Epoch 008 train=0.74638 valEMD=119.1034 lr=3.10e-06 BEST
Epoch 009 train=0.74721 valEMD=117.6689 lr=3.10e-06 BEST
Epoch 010 train=0.74269 valEMD=116.3516 lr=3.10e-06 BEST
Epoch 011 train=0.74468 valEMD=114.9082 lr=3.10e-06 BEST
Epoch 012 train=0.73254 valEMD=113.6962 lr=3.10e-06 BEST
Epoch 013 train=0.72817 valEMD=112.4763 lr=3.10e-06 BEST
Epoch 014 train=0.72480 valEMD=111.1334 lr=3.10e-06 BEST
Epoch 015 train=0.70698 valEMD=109.9791 lr=3.10e-06 BEST
Epoch 016 train=0.715

KeyboardInterrupt: 

# 2

In [26]:
# ============================================================
# DINOv2 + FLEXIBLE WEIBULL + SAM 2.1 COARSE-PARTICLE CORRECTION
#
# RUN THIS CELL AFTER THE DINOv2 PIPELINE HAS COMPLETED.
#
# Uses existing DINO pipeline variables when available:
#   ts / pred_A : sample-level test PSD predictions
#   labels      : labelled training PSD table indexed by sample_id
#   train_meta, test_meta : metadata tables
#   CFG, OUT, SIZE_COLS
#
# IMPORTANT:
# 1. SAM measures visible image area, not mass fraction.
# 2. A one-sample calibration (H374) is exploratory, not robust.
# 3. Accurate pixel/mm calibration is required. This cell refuses
#    to silently assume a physical scale when it is unavailable.
# ============================================================

import gc
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. ADAPT VARIABLES FROM THE DINOv2 PIPELINE
# ------------------------------------------------------------
SAM2_MODEL = "facebook/sam2.1-hiera-large"

if "CFG" not in globals():
    CFG = {}

CFG.setdefault("work_ppm", 2.0)
CFG.setdefault("work_crop", 0.02)
CFG.setdefault("points_per_batch", 64)
CFG.setdefault("pred_iou_thresh", 0.84)
CFG.setdefault("stability_score_thresh", 0.88)
CFG.setdefault("min_side_mm", 1.0)
CFG.setdefault("max_area_frac", 0.25)
CFG.setdefault("overlap_drop", 0.70)
CFG.setdefault("min_fill", 0.18)
CFG.setdefault("apply_if_A63_ge", 0.02)
CFG.setdefault("sam_min_ppm", 0.0)

# Set this only if your dataset has a known, uniform pixel/mm scale.
# Example: CFG["sam_default_ppm"] = 12.5
# It is intentionally NOT guessed here.
CFG.setdefault("sam_default_ppm", None)

if "SIZE_COLS" not in globals():
    SIZE_COLS = ["0.002", "0.0063", "0.02", "0.063", "0.2",
                 "0.63", "2", "6.3", "20", "63", "200"]

if "OUT" in globals():
    OUTPUT_DIR = Path(OUT)
elif "OUTPUT_DIR" in globals():
    OUTPUT_DIR = Path(OUTPUT_DIR)
else:
    OUTPUT_DIR = Path("/kaggle/working/dinov2_dog5_psd_outputs_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DINO pipeline stores sample-level test predictions in `ts`.
if "pred_A" not in globals():
    if "ts" in globals():
        pred_A = ts.copy()
    else:
        saved_pred = OUTPUT_DIR / "test_sample_predictions.csv"
        if not saved_pred.exists():
            raise NameError(
                "Neither `pred_A` nor `ts` exists, and "
                f"{saved_pred} was not found. Run the DINOv2 pipeline first."
            )
        pred_A = pd.read_csv(saved_pred, index_col=0)

pred_A = pred_A.copy()
pred_A.index = pred_A.index.astype(str).str.strip()
pred_A = pred_A.reindex(columns=SIZE_COLS)

if pred_A.empty or pred_A.isna().all(axis=None):
    raise ValueError("`pred_A` is empty or has no usable PSD predictions.")
if pred_A[SIZE_COLS].isna().any().any():
    raise ValueError(
        "`pred_A` is missing one or more PSD values or target columns. "
        f"Expected columns: {SIZE_COLS}"
    )

if "labels" not in globals():
    raise NameError(
        "`labels` is not defined. Run the DINOv2 training/label-loading cell first."
    )
lab = labels.copy()
lab.index = lab.index.astype(str).str.strip()
lab = lab.reindex(columns=SIZE_COLS)

if "train_meta" not in globals() or "test_meta" not in globals():
    raise NameError(
        "`train_meta` and `test_meta` are required to build the photo table. "
        "Run the DINOv2 metadata-loading cell first, or define `photos` with "
        "path, split, sample_id, and ppm_eff columns."
    )

# ------------------------------------------------------------
# 1. BUILD / NORMALIZE THE SOURCE PHOTO TABLE
# ------------------------------------------------------------
# Prefer a previously prepared `photos` table because it may contain
# the original physical calibration `ppm_eff`.
required_photo_cols = {"path", "split", "sample_id", "ppm_eff"}

if "photos" in globals() and isinstance(photos, pd.DataFrame) and required_photo_cols.issubset(photos.columns):
    photo_table = photos.copy()
else:
    def _metadata_to_photo_table(meta, split_name):
        d = meta.copy()
        if "image_path" not in d.columns or "sample_id" not in d.columns:
            raise ValueError(
                f"{split_name}_meta must contain image_path and sample_id."
            )

        # Look for a physical image scale in metadata.
        ppm_col = next(
            (c for c in ["ppm_eff", "pixels_per_mm", "px_per_mm", "ppm"]
             if c in d.columns),
            None
        )
        if ppm_col is not None:
            d["ppm_eff"] = pd.to_numeric(d[ppm_col], errors="coerce")
        elif CFG["sam_default_ppm"] is not None:
            d["ppm_eff"] = float(CFG["sam_default_ppm"])
        else:
            raise ValueError(
                f"{split_name}_meta has no ppm_eff/pixels_per_mm/px_per_mm/ppm column. "
                "SAM particle sizes in millimetres cannot be estimated safely without "
                "a physical scale. Define photos with ppm_eff or set "
                "CFG['sam_default_ppm'] to a known calibrated value."
            )

        d["path"] = d["image_path"].astype(str)
        d["split"] = split_name
        d["sample_id"] = d["sample_id"].astype(str).str.strip()

        # Feature metadata can contain multiple augmentation rows per source photo.
        # Deduplicate by physical source path so SAM does not segment the same
        # original photo repeatedly.
        keep = ["path", "split", "sample_id", "ppm_eff"]
        return d[keep].drop_duplicates(subset=["split", "path"]).copy()

    photo_table = pd.concat(
        [
            _metadata_to_photo_table(train_meta, "train"),
            _metadata_to_photo_table(test_meta, "test"),
        ],
        ignore_index=True,
    )

photo_table["path"] = photo_table["path"].astype(str)
photo_table["split"] = photo_table["split"].astype(str).str.strip().str.lower()
photo_table["split"] = photo_table["split"].replace({
    "training": "train", "trainset": "train", "validation": "train",
    "testing": "test", "testset": "test",
})
photo_table["sample_id"] = photo_table["sample_id"].astype(str).str.strip()
photo_table["ppm_eff"] = pd.to_numeric(photo_table["ppm_eff"], errors="coerce")

# Remove invalid/missing paths and invalid scales.
photo_table = photo_table[
    photo_table["path"].map(lambda p: Path(p).is_file())
    & np.isfinite(photo_table["ppm_eff"])
    & (photo_table["ppm_eff"] > 0)
].copy()

if photo_table.empty:
    raise RuntimeError(
        "No existing source images with valid paths and physical ppm_eff were found. "
        "Check metadata paths and the image-scale column."
    )

# Keep one row per source photo and split; avoid duplicate augmentation entries.
photo_table = photo_table.drop_duplicates(subset=["split", "path"]).reset_index(drop=True)

# Ensure H374 remains available for calibration even if it is below the usual
# high-resolution threshold.
min_ppm = float(CFG["sam_min_ppm"])
targets = photo_table[
    (photo_table["ppm_eff"] >= min_ppm)
    | ((photo_table["split"] == "train") & (photo_table["sample_id"] == "H374"))
].copy()

if not (targets["sample_id"].eq("H374") & targets["split"].eq("train")).any():
    raise ValueError(
        "No H374 training photo is available for calibration. "
        "Ensure the source photo table includes H374 and its physical scale."
    )

print("=" * 80)
print("DINOv2 + SAM 2.1 COARSE-PARTICLE CORRECTION")
print("=" * 80)
print("SAM model:", SAM2_MODEL)
print("Output directory:", OUTPUT_DIR)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
print("Source photos available:", len(photo_table))
print("Photos selected for SAM:", len(targets))
print("Split counts:", targets["split"].value_counts().to_dict())
print("Samples selected:", targets["sample_id"].nunique())

# ------------------------------------------------------------
# 2. LOAD SAM 2.1 AUTOMATIC MASK GENERATOR
# ------------------------------------------------------------
sam_device = 0 if torch.cuda.is_available() else -1
print("\nLoading SAM 2.1 Hiera-Large...")
load_start = time.time()
mask_gen = pipeline(
    "mask-generation",
    model=SAM2_MODEL,
    device=sam_device,
)
print(f"SAM 2.1 loaded in {time.time() - load_start:.1f} seconds")

# ------------------------------------------------------------
# 3. IMAGE PREPARATION
# ------------------------------------------------------------
def load_work(path, ppm):
    """EXIF-correct, rescale to work_ppm, landscape-orient, crop borders."""
    with Image.open(path) as src:
        im = ImageOps.exif_transpose(src).convert("RGB")

    w, h = im.size
    factor = float(CFG["work_ppm"]) / float(ppm)
    new_w = max(1, round(w * factor))
    new_h = max(1, round(h * factor))

    arr = cv2.resize(
        np.asarray(im),
        (new_w, new_h),
        interpolation=cv2.INTER_AREA if factor < 1 else cv2.INTER_LINEAR,
    )

    if arr.shape[0] > arr.shape[1]:
        arr = np.ascontiguousarray(np.rot90(arr))

    H, W = arr.shape[:2]
    crop = float(CFG["work_crop"])
    y0, y1 = int(H * crop), int(H * (1.0 - crop))
    x0, x1 = int(W * crop), int(W * (1.0 - crop))
    arr = arr[y0:y1, x0:x1]

    if arr.size == 0:
        raise ValueError(f"Image became empty after cropping: {path}")

    return np.ascontiguousarray(arr).astype(np.uint8)

# ------------------------------------------------------------
# 4. MASKS -> STONE-LIKE OBJECTS
# ------------------------------------------------------------
def stones_from_masks(masks, scores, total_pixels):
    if not masks:
        return [], 0.0, np.zeros((1, 1), dtype=np.int32)

    valid_masks, valid_scores = [], []
    for mask, score in zip(masks, scores):
        m = np.asarray(mask)
        # Hugging Face mask-generation output may contain PIL images.
        if m.ndim == 3:
            m = np.squeeze(m)
        if m.ndim != 2:
            continue
        m = m.astype(bool)
        if not m.any():
            continue
        valid_masks.append(m)
        valid_scores.append(float(score))

    if not valid_masks:
        shape = np.asarray(masks[0]).squeeze().shape
        if len(shape) != 2:
            shape = (1, 1)
        return [], 0.0, np.zeros(shape, dtype=np.int32)

    ref_shape = valid_masks[0].shape
    aligned_masks, aligned_scores = [], []
    for m, score in zip(valid_masks, valid_scores):
        if m.shape != ref_shape:
            m = cv2.resize(
                m.astype(np.uint8),
                (ref_shape[1], ref_shape[0]),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)
        aligned_masks.append(m)
        aligned_scores.append(score)

    order = sorted(
        range(len(aligned_masks)),
        key=lambda i: -int(aligned_masks[i].sum()),
    )
    taken = np.zeros(ref_shape, dtype=bool)
    label_img = np.zeros(ref_shape, dtype=np.int32)
    stones = []
    min_area = (float(CFG["min_side_mm"]) * float(CFG["work_ppm"])) ** 2

    for i in order:
        m = aligned_masks[i]
        score = aligned_scores[i]
        raw_size = int(m.sum())

        if raw_size < min_area:
            continue
        if raw_size > float(CFG["max_area_frac"]) * float(total_pixels):
            continue

        overlap = int((m & taken).sum())
        if overlap > float(CFG["overlap_drop"]) * raw_size:
            continue

        # Use only pixels not already assigned to a previously accepted mask.
        unique = m & (~taken)
        unique_size = int(unique.sum())
        if unique_size < min_area:
            continue

        ys, xs = np.nonzero(unique)
        if len(xs) < 3:
            continue

        pts = np.column_stack([xs, ys]).astype(np.float32)
        pts -= pts.mean(axis=0, keepdims=True)

        try:
            eig = np.linalg.eigvalsh(np.cov(pts.T))
            largest = max(float(eig.max()), 1e-9)
        except Exception:
            largest = 0.0

        major_mm = 4.0 * np.sqrt(max(largest, 1e-9)) / float(CFG["work_ppm"])
        ellipse_area_px = np.pi * (major_mm * float(CFG["work_ppm"]) / 2.0) ** 2
        fill = unique_size / max(float(ellipse_area_px), 1e-9)

        if fill < float(CFG["min_fill"]):
            continue

        taken |= unique
        label_img[unique] = len(stones) + 1
        stones.append({
            "area_frac": unique_size / float(total_pixels),
            "major_mm": float(major_mm),
            "score": float(score),
            "pixels": unique_size,
            "fill": float(fill),
        })

    return stones, float(taken.mean()), label_img

# ------------------------------------------------------------
# 5. RUN SAM 2.1 ON TRAIN CALIBRATION + TEST PHOTOS
# ------------------------------------------------------------
measurements = []
keep_for_plot = {}
failed_images = []
start = time.time()

for row in tqdm(targets.itertuples(index=False), total=len(targets),
                desc="SAM 2.1 segmentation", unit="photo"):
    path = Path(row.path)
    sid = str(row.sample_id)
    split = str(row.split)
    try:
        rgb = load_work(path, float(row.ppm_eff))
        output = mask_gen(
            Image.fromarray(rgb),
            points_per_batch=int(CFG["points_per_batch"]),
            pred_iou_thresh=float(CFG["pred_iou_thresh"]),
            stability_score_thresh=float(CFG["stability_score_thresh"]),
        )

        masks = [np.asarray(m).squeeze() for m in output.get("masks", [])]
        raw_scores = output.get("scores", [])
        if torch.is_tensor(raw_scores):
            raw_scores = raw_scores.detach().cpu().numpy()
        scores = [float(s) for s in raw_scores]
        if len(scores) != len(masks):
            scores = [1.0] * len(masks)

        stones, cover, label_img = stones_from_masks(
            masks, scores, rgb.shape[0] * rgb.shape[1]
        )

        A63 = float(sum(s["area_frac"] for s in stones if s["major_mm"] > 63.0))
        A20 = float(sum(s["area_frac"] for s in stones if s["major_mm"] > 20.0))
        max_stone_mm = max((s["major_mm"] for s in stones), default=0.0)
        mean_score = float(np.mean([s["score"] for s in stones])) if stones else 0.0

        measurements.append({
            "file": path.name,
            "path": str(path),
            "split": split,
            "sample_id": sid,
            "ppm_eff": float(row.ppm_eff),
            "cover": cover,
            "n_stones": len(stones),
            "A63": A63,
            "A20": A20,
            "max_stone_mm": max_stone_mm,
            "mean_stone_score": mean_score,
        })

        # Keep the first image per split/sample for visual review.
        key = (split, sid)
        if key not in keep_for_plot:
            keep_for_plot[key] = (rgb, label_img)

    except Exception as exc:
        failed_images.append({
            "path": str(path), "split": split, "sample_id": sid,
            "error": f"{type(exc).__name__}: {exc}",
        })
        print(f"\nSAM failed on {path.name}: {type(exc).__name__}: {exc}")

if not measurements:
    raise RuntimeError(
        "SAM 2.1 produced no usable measurements. Check image paths, scale, "
        "Transformers version, and GPU memory."
    )

meas = pd.DataFrame(measurements)
meas.to_csv(OUTPUT_DIR / "sam21_stone_measurements.csv", index=False)
if failed_images:
    pd.DataFrame(failed_images).to_csv(
        OUTPUT_DIR / "sam21_failed_images.csv", index=False
    )

print(f"\nSAM segmentation finished in {time.time() - start:.1f} seconds.")
print("Successful photo measurements:", len(meas))
print("Failed photos:", len(failed_images))

# ------------------------------------------------------------
# 6. AGGREGATE BY SAMPLE AND SPLIT
# ------------------------------------------------------------
ms = (
    meas.groupby(["split", "sample_id"], sort=False)
    .agg(
        photos=("file", "size"),
        A63=("A63", "mean"),
        A20=("A20", "mean"),
        cover=("cover", "mean"),
        max_stone_mm=("max_stone_mm", "max"),
        mean_stone_score=("mean_stone_score", "mean"),
    )
)

ms.to_csv(OUTPUT_DIR / "sam21_sample_stone_statistics.csv")
print("\nSAM 2.1 stone measurements:")
print(ms.round(4).sort_values("A20", ascending=False).to_string())

# H374 calibration must come from the training split.
cal_key = ("train", "H374")
if cal_key not in ms.index:
    raise ValueError(
        "H374 was not successfully measured in the train split; cannot calibrate."
    )
if "H374" not in lab.index:
    raise ValueError("H374 is absent from the labelled PSD table.")
if "20" not in lab.columns or "63" not in lab.columns:
    raise ValueError("Label table must contain PSD columns '20' and '63'.")

h374_a20 = float(ms.loc[cal_key, "A20"])
if not np.isfinite(h374_a20) or h374_a20 <= 1e-8:
    raise ValueError(
        f"H374 A20={h374_a20} is zero/invalid. Cannot estimate a stable K_CAL."
    )

true_gt20 = 100.0 - float(lab.loc["H374", "20"])
K_CAL = true_gt20 / (100.0 * h374_a20)

print("\n" + "=" * 80)
print("SAM 2.1 COARSE-FRACTION CALIBRATION")
print("=" * 80)
print(f"K_CAL: {K_CAL:.6f}")
print(f"H374 true >20 mm fraction: {true_gt20:.3f}%")
print(f"H374 measured A20: {100.0 * h374_a20:.3f}%")
print(f"H374 true >63 mm fraction: {100.0 - float(lab.loc['H374', '63']):.3f}%")
print(f"H374 measured A63: {100.0 * float(ms.loc[cal_key, 'A63']):.3f}%")
print("Caution: a single-sample calibration is exploratory only.")

# ------------------------------------------------------------
# 7. PSD POSTPROCESSING AND CORRECTION
# ------------------------------------------------------------
I20 = SIZE_COLS.index("20")
I63 = SIZE_COLS.index("63")

def _postprocess_psd(curves):
    curves = np.clip(np.asarray(curves, dtype=float).copy(), 0.0, 100.0)
    curves = np.maximum.accumulate(curves, axis=-1)
    curves[..., -1] = 100.0
    return curves

def redraw(curve, a63, a20):
    c = np.asarray(curve, dtype=float).copy()

    # Convert the visible-area estimates into exploratory coarse PSD estimates.
    new20 = float(np.clip(100.0 - K_CAL * 100.0 * float(a20), 0.0, 100.0))
    new63 = float(np.clip(100.0 - K_CAL * 100.0 * float(a63), new20, 100.0))

    # Preserve the fine-region shape, scaling it only when the 20-mm point drops.
    old20 = max(float(c[I20]), 1e-6)
    scale = new20 / old20
    if scale < 1.0:
        c[:I20 + 1] *= scale

    c[I20] = new20
    c[I63] = new63
    c[-1] = 100.0
    return _postprocess_psd(c)

# Only test photos/samples are corrected. Training labels are used only for
# the H374 calibration; they are never overwritten by the correction.
test_ms = ms.xs("test", level="split") if "test" in ms.index.get_level_values("split") else pd.DataFrame()
final_rows = []
changed = []

for sid in pred_A.index.astype(str):
    base = pred_A.loc[sid, SIZE_COLS].to_numpy(dtype=float)

    if sid in test_ms.index:
        a63 = float(test_ms.loc[sid, "A63"])
        a20 = float(test_ms.loc[sid, "A20"])
    else:
        a63, a20 = 0.0, 0.0

    if a63 >= float(CFG["apply_if_A63_ge"]):
        corrected = redraw(base, a63, a20)
        changed.append(sid)
    else:
        corrected = _postprocess_psd(base)

    final_rows.append(corrected)

pred_B = pd.DataFrame(
    np.stack(final_rows),
    index=pred_A.index,
    columns=SIZE_COLS,
)
pred_B = pd.DataFrame(
    _postprocess_psd(pred_B.to_numpy()),
    index=pred_B.index,
    columns=SIZE_COLS,
)

print("\n" + "=" * 80)
print("SAM 2.1 PSD CORRECTION")
print("=" * 80)
print("Test samples changed:", changed)

for sid in changed:
    comparison = pd.DataFrame({
        "DINO_Flexible_Weibull": pred_A.loc[sid, SIZE_COLS],
        "DINO_plus_SAM2.1": pred_B.loc[sid, SIZE_COLS],
    }).T
    print(f"\n{sid}")
    print(comparison.round(2).to_string())

pred_B_path = OUTPUT_DIR / "pred_B_dino_flexible_weibull_plus_sam21.csv"
pred_B.to_csv(pred_B_path, index_label="sample_id")

# Also create a submission-shaped file when the original template is available.
if "submission_template" in globals():
    template = submission_template.copy()
    template["sample_id"] = template["sample_id"].astype(str).str.strip()
    lookup = {str(s).strip(): s for s in pred_B.index.astype(str)}
    canon = lambda x: " ".join(str(x).strip().replace("_", " ").split())
    canonical_lookup = {canon(s): s for s in pred_B.index.astype(str)}

    rows = []
    missing = []
    for sid in template["sample_id"]:
        actual = lookup.get(sid, canonical_lookup.get(canon(sid)))
        if actual is None:
            missing.append(sid)
        else:
            rows.append(pred_B.loc[actual, SIZE_COLS].to_numpy(float))

    if missing:
        raise ValueError(f"Corrected PSD missing submission IDs: {missing}")

    submission_B = template[["sample_id"]].copy()
    submission_B[SIZE_COLS] = _postprocess_psd(np.asarray(rows, dtype=float))
    submission_B_path = OUTPUT_DIR / "submission_dino_plus_sam21.csv"
    submission_B.to_csv(submission_B_path, index=False)
    print("Corrected submission:", submission_B_path)

print("\nSaved measurement file:", OUTPUT_DIR / "sam21_stone_measurements.csv")
print("Saved sample statistics:", OUTPUT_DIR / "sam21_sample_stone_statistics.csv")
print("Saved corrected PSD:", pred_B_path)
print("Final corrected dataframe variable: pred_B")

del mask_gen
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nSAM 2.1 branch complete.")


DINOv2 + SAM 2.1 COARSE-PARTICLE CORRECTION
SAM model: facebook/sam2.1-hiera-large
Output directory: /kaggle/working/dinov2_dog5_psd_outputs_v2
Device: GPU
Source photos available: 162
Photos selected for SAM: 162
Split counts: {'train': 127, 'test': 35}
Samples selected: 34

Loading SAM 2.1 Hiera-Large...


Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

SAM 2.1 loaded in 1.8 seconds


SAM 2.1 segmentation:   0%|          | 0/162 [00:00<?, ?photo/s]


SAM segmentation finished in 1008.4 seconds.
Successful photo measurements: 162
Failed photos: 0

SAM 2.1 stone measurements:
                                photos     A63     A20   cover  max_stone_mm  mean_stone_score
split sample_id                                                                               
test  HPC_Muenster_BS6_9_0-10m       5  0.2514  0.6113  0.7112      104.1951            0.9278
      HPC_Testfeld Lidl WHV          4  0.0000  0.3273  0.4405       46.9640            0.9171
train H374                           3  0.0198  0.1470  0.2738       78.1611            0.9058
      H372                           6  0.0000  0.0695  0.1817       43.3936            0.9056
      H371                           7  0.0000  0.0601  0.1824       42.1435            0.9034
      H366                           4  0.0000  0.0574  0.1512       38.7493            0.9050
      H183                           7  0.0000  0.0491  0.1469       42.5310            0.9023
      H181        